# 03 · Python III: Errors, Files, Classes & Modules

This is the hands-on companion to [`index.html`](index.html) in this folder. It assumes you have
completed **Module 01** and **Module 02** and nothing more. You have never seen a class before;
by the end of this notebook you will have written four.

By the end, you will be able to:

- read a traceback bottom-up and tell the crash site apart from the root cause;
- recognise the eleven exceptions you will actually meet, by their message;
- handle expected failure with `try` / `except` / `else` / `finally`;
- raise your own named exceptions, chain them with `from`, and know when `assert` is wrong;
- find a file from any working directory using `pathlib`;
- read and write text, CSV, and JSON without losing data or mangling characters;
- define a class with `__init__`, `self`, and `__repr__`, and explain what `self` is;
- build a scikit-learn-shaped model with `fit`, `predict`, `mean_`, and `NotFittedError`; and
- import your own module and reach for the standard library before writing a loop.

**Working rule, unchanged from Module 02:** predict first, run second, explain third. A wrong
prediction is useful - it shows exactly which mental model needs adjusting.

## 0 · How to use this notebook

Click a code cell and press **Shift + Enter** to run it and move to the next one. Output appears
below the cell. If the notebook starts behaving strangely because cells were run out of order, use
**Kernel -> Restart Kernel and Run All Cells**.

Run it from top to bottom the first time. No third-party packages are required anywhere: everything
here comes from Python's own standard library.

> Cells marked **Predict before running** contain a comment line for your guess. Fill it in before
> you run the cell. That habit is the single fastest way to learn.

**Everything this notebook writes to disk goes into one folder called `sandbox/`,** created in the
next cell. Nothing is ever written next to the notebook or above it, so you can delete `sandbox/`
at any time and rerun.

### Companion map

| Lesson part on the page | Section here |
|---|---|
| Part 1 · Read the error | 1, 2 |
| Part 2 · Handle what you expect | 3 |
| Part 3 · Fail well, then find the cause | 4 |
| Part 4 · Find the file | 5 |
| Part 5 · Read and write real data | 6, 7 |
| Part 6 · Objects: data with behaviour | 8 |
| Part 7 · Reuse and shape | 9 |
| Part 8 · Modules, packages, and the standard library | 10 |
| Part 9 · Build the study tracker | 12 |
| Part 10 · Keep a reference | 11, 13, 14, 15 |

Read a lesson part first, then run the matching section here.

In [ ]:
# Setup. Run this before anything else.
from pathlib import Path

SANDBOX = Path("sandbox")          # a RELATIVE path: it resolves against the working directory
SANDBOX.mkdir(exist_ok=True)       # exist_ok=True means "fine if it is already there"

print("working directory:", Path.cwd())
print("sandbox folder   :", SANDBOX.resolve())
print("sandbox exists?  ", SANDBOX.exists())

`Path.cwd()` is the **working directory**: the folder Python considers "here". `SANDBOX.resolve()`
turned the relative path `sandbox` into an absolute one by joining it to that folder. Section 5 is
entirely about this, because "why can't Python find my file" is almost always this and nothing else.

### Three helpers, defined once

Module 02 gave you `check`. This module needs two more, because you are about to spend a lot of time
with things that fail on purpose:

- `check(name, actual, expected)` - prints **PASS** or **FAIL** for one expected value. Numbers are
  compared within a small `tolerance`, because `0.1 + 0.2` is not exactly `0.3`.
- `show_error(func, *args, **kwargs)` - calls `func` and **prints the exception type and message
  instead of crashing the notebook**. This is how you meet an exception on purpose without stopping
  the run.
- `check_raises(name, exception_type, func, ...)` - prints **PASS** if calling `func` raises
  `exception_type` (or a subclass of it). This is what makes an exercise about failure gradable.

Read the three definitions before running them. Everything in them is Module 02 material: `def`,
default arguments, `*args` and `**kwargs`, `try`/`except`, and f-strings.

In [ ]:
def check(name, actual, expected, tolerance=1e-9):
    """Print PASS/FAIL for one expected value. Numbers compare within tolerance."""
    if actual is None and expected is not None:
        print(f"--    {name}: not attempted yet")
    elif isinstance(expected, float) and isinstance(actual, (int, float)):
        if abs(actual - expected) <= tolerance:
            print(f"PASS  {name}")
        else:
            print(f"FAIL  {name}: got {actual!r}; expected {expected!r}")
    elif actual == expected:
        print(f"PASS  {name}")
    else:
        print(f"FAIL  {name}: got {actual!r}; expected {expected!r}")


def show_error(func, *args, **kwargs):
    """Call func and print the exception type and message instead of crashing the notebook.

    Use this to meet an exception on purpose. Returns func's value when nothing went
    wrong, and None when an exception was caught and printed.
    """
    try:
        result = func(*args, **kwargs)
    except Exception as error:
        print(f"{type(error).__name__}: {error}")
        return None
    else:
        print(f"no exception - the call returned {result!r}")
        return result


def check_raises(name, exception_type, func, *args, **kwargs):
    """Print PASS if calling func raises exception_type (or a subclass), FAIL otherwise."""
    try:
        result = func(*args, **kwargs)
    except exception_type as error:
        print(f"PASS  {name}: raised {type(error).__name__}: {error}")
    except Exception as error:
        print(f"FAIL  {name}: raised {type(error).__name__}, expected {exception_type.__name__}")
    else:
        print(f"FAIL  {name}: nothing raised; the call returned {result!r}")


# Self-tests: three outcomes each, so you know what the output looks like.
check("check · a match", 4, 4)
check("check · not attempted", None, 4)
check("check · a miss", 5, 4)
print()
show_error(int, "thirty")
show_error(int, "30")
print()
check_raises("check_raises · correct type", ValueError, int, "thirty")
check_raises("check_raises · wrong type", KeyError, int, "thirty")
check_raises("check_raises · no error", ValueError, int, "30")

Nine lines of output, and each one is worth recognising now rather than later:

- `--` means the answer is still `None`, so nothing was attempted. Every exercise below stays safe to
  run in this state, which is why you can do **Restart Kernel and Run All Cells** at any time.
- `show_error` printed `ValueError: invalid literal for int() with base 10: 'thirty'` and the
  notebook carried on. That line is the exception's **type** and its **message**, which is exactly
  what the last line of a traceback shows you.
- `check_raises` distinguishes three different things: the right exception, the wrong exception, and
  no exception at all. "It failed" is not the same as "it failed correctly".

These three names are used everywhere below, and in the lesson page's labs. They are the only
machinery this notebook adds.

## 1 · Crash on purpose, and read the wreckage

An **exception** is Python's way of saying "I cannot continue, and here is exactly why". A
**traceback** is the report it prints: a list of the function calls that were in progress, ending
with the exception itself.

Three words tell you how to read one:

- **what** - the last line: the exception type and message;
- **where** - the frames above it, innermost last;
- **why** - your own newest frame, which is usually one step *above* the crash.

The rule to memorise: **read a traceback bottom-up.** The last line is the only line guaranteed to
be about your problem.

### This cell errors on purpose - that is the point

`average([])` divides by zero. Below, the crash is caught only so that the notebook can print the
traceback and keep running: `traceback.print_exc()` prints the *same* report Jupyter would show you.
The last line of the cell is the raw, uncaught version - **uncomment it** and run the cell again to
see Jupyter's own rendering, with the `----> ` arrow pointing at the failing line.

In [ ]:
import traceback


def average(values):
    """Return the mean of values."""
    total = sum(values)
    return total / len(values)


scores = []                        # oops -- empty
try:
    result = average(scores)
except ZeroDivisionError:
    traceback.print_exc()          # prints the traceback WITHOUT stopping the notebook

# Uncomment the next line and run the cell again to see the real, uncaught crash:
# result = average(scores)

Read what was printed bottom-up.

1. **What:** `ZeroDivisionError: division by zero`.
2. **Where:** the frame just above names the line `return total / len(values)`, inside `average`.
3. **Why:** the frame above *that* is the call `average(scores)` - and `scores` was empty.

The division is the **crash site**. The empty list is the **root cause**. They are on different
lines, written at different times, and only one of them is a mistake: dividing by `len(values)` is
perfectly reasonable code. This gap is the single most valuable debugging idea in the module, and it
is why Python prints the whole chain instead of just the failing line.

### Predict before running · which line is the mistake?

The next cell puts three frames between your call and the crash. Before running it, write down which
line you think is the actual mistake - the line where a *wrong value* was created, not the line
where Python finally gave up.

In [ ]:
# My prediction: the mistake is on the line that says ______________________


def report_for(minutes_by_name, name):
    """Return the average minutes for one learner."""
    chosen = minutes_by_name.get(name, [])      # a missing name quietly becomes []
    return average(chosen)


def print_all(minutes_by_name, names):
    """Print a one-line report for each requested name."""
    for name in names:
        print(name, "->", report_for(minutes_by_name, name))


log = {"maya": [45, 30, 60]}

try:
    print_all(log, ["maya", "ravi"])
except ZeroDivisionError:
    traceback.print_exc()

Four frames now: `print_all` -> `report_for` -> `average` -> the division. The crash is still on the
division, and it is still not the mistake.

The mistake is `minutes_by_name.get(name, [])`. Asking for `"ravi"`, who is not in the log, produced
an empty list, and an empty list is a perfectly valid thing to pass to a function that will divide by
its length. **Scan upward from the bottom until you reach a frame in code you wrote, then look at the
values that line handed onward.** Those values are your suspects.

Notice also that `"maya"` printed successfully first. A traceback tells you about the call that
failed, not about the ones that worked.

### Chained tracebacks

Sometimes a traceback contains *two* reports joined by a sentence:

```
The above exception was the direct cause of the following exception
```

That sentence is produced by `raise NewError(...) from original_error`, which Section 4 teaches. It
means: something failed, and the code deliberately replaced it with a more meaningful failure while
keeping the original attached.

In [ ]:
def to_minutes(text):
    """Convert text to an int, or fail with a message that names the offending value."""
    try:
        return int(text)
    except ValueError as error:
        raise RuntimeError("bad minutes value: " + repr(text)) from error


try:
    to_minutes("thirty")
except RuntimeError:
    traceback.print_exc()

Two blocks, one bridge sentence.

- The **first** block is the origin: `ValueError: invalid literal for int() with base 10: 'thirty'`.
  Read this block to find out what actually went wrong.
- The bridge sentence tells you the second block exists because a human chose to re-raise.
- The **last line** is what escaped and what the caller had to handle: `RuntimeError: bad minutes
  value: 'thirty'`.

So: read the **first** block for the cause, the **last** line for what escaped. A related sentence,
`During handling of the above exception, another exception occurred`, means the second failure
happened *by accident* inside an `except` block - that one is usually a bug in your error handling.

### Tracebacks in a notebook

Jupyter prints the same information with three cosmetic differences:

```
---------------------------------------------------------------------------
ZeroDivisionError                         Traceback (most recent call last)
Cell In[12], line 6
      4 return total / len(values)
      5 scores = []
----> 6 result = average(scores)

ZeroDivisionError: division by zero
```

- `Cell In[12]` is the cell's execution number, not a file. `In [12]` in the margin is the same cell.
- `----> 6` points at the failing line **within that cell**, and Jupyter shows a few lines of
  context around it.
- The **top** frame is your cell, because your cell is what the kernel called. In a script the top
  frame is the file's top level. Either way, the reading order is unchanged: bottom-up.

Common ways to waste an afternoon: scrolling past the traceback straight back to the code; guessing
instead of reading the last line; and assuming the deepest frame is your bug when it is three
directories inside a library you did not write.

## 2 · The exceptions you will meet

Eleven names cover almost everything a beginner hits. Naming the failure is most of the fix, so this
section triggers every one of them **deliberately**, using `show_error` so the notebook survives.

| Exception | It means | First thing to check |
|---|---|---|
| `NameError` | this name has never been assigned | a typo, or use before assignment |
| `TypeError` | wrong **kind** of thing | what `type()` says about each operand |
| `ValueError` | right kind, wrong **content** | the actual value, printed with `repr` |
| `IndexError` | no such position | `len()` - the last index is `len - 1` |
| `KeyError` | no such key | `in`, or `.get()` with a fallback |
| `AttributeError` | no such attribute or method | spelling, and whether anything assigned it yet |
| `ZeroDivisionError` | divided by zero | an empty collection is the usual cause |
| `FileNotFoundError` | no file at that path | the working directory, not the file |
| `ModuleNotFoundError` | not installed *in this environment* | `sys.executable` |
| `IndentationError` / `SyntaxError` | Python could not even read the code | the line above the caret |
| `UnboundLocalError` | a local name used before its assignment | assignment anywhere makes it local |

Three tiny helpers make the demonstrations short. Each does exactly one risky thing, so
`show_error` has something to call.

In [ ]:
def get_index(values, position):
    """Return values[position]."""
    return values[position]


def get_key(mapping, key):
    """Return mapping[key]."""
    return mapping[key]


def get_attribute(thing, attribute_name):
    """Return thing.attribute_name."""
    return getattr(thing, attribute_name)


# 1 · NameError -- nothing has ever been assigned to this name.
def use_a_missing_name():
    return scoers + 1              # 'scores' misspelt


show_error(use_a_missing_name)

# 2 · TypeError -- the wrong KIND of thing.
def label_the_minutes():
    return "minutes: " + 45        # str + int


show_error(label_the_minutes)

`NameError: name 'scoers' is not defined` is Python's most literal message: it looked the name up and
found nothing. The two causes are a typo and using a name before the line that assigns it.

`TypeError: can only concatenate str (not "int") to str` is about **kinds**. `+` means "join" for
strings and "add" for numbers, and Python refuses to guess which one you meant. The fix is a
conversion at the boundary: `"minutes: " + str(45)`, or better, an f-string.

In [ ]:
# 3 · ValueError -- the right kind of thing, with content that cannot work.
show_error(int, "thirty")

# 4 · IndexError -- there is no such position.
scores = [90, 85, 77]
print("len(scores) is", len(scores), "so the last valid index is", len(scores) - 1)
show_error(get_index, scores, 3)

# 5 · KeyError -- there is no such key.
totals = {"maya": 135, "ravi": 60, "ada": 165}
show_error(get_key, totals, "bob")
print("the safe way:", totals.get("bob"), "and with a fallback:", totals.get("bob", 0))

`int("thirty")` is a `ValueError`, not a `TypeError`: a string is exactly the kind of thing `int()`
accepts, so the complaint is about the content. `int("30")` works.

`IndexError: list index out of range` is almost always off-by-one. A list of length 3 has indices
`0`, `1`, `2`. Print `len()` before you argue with it.

`KeyError: 'bob'` prints the missing key, which is often enough to spot a stray space or a
capitalisation difference. `.get(key)` returns `None` instead of raising, and `.get(key, fallback)`
returns your fallback - but only reach for them when a missing key is genuinely acceptable.

In [ ]:
# 6 · AttributeError -- no such attribute or method on this object.
class Draft:
    """An object with no attributes yet, to make the point."""


show_error(get_attribute, Draft(), "mean_")
show_error(get_attribute, "maya", "push")          # str has .append? no -- and no .push either

# 7 · ZeroDivisionError -- an empty collection is the usual cause.
show_error(average, [])
print("with data:", average([90, 85, 77]))

`AttributeError: 'Draft' object has no attribute 'mean_'` is a story about **order**, not spelling,
about half the time: the attribute does not exist *yet* because no line has assigned it. That is
exactly what happens when you call `predict()` before `fit()` - Section 9 turns that accident into a
deliberate, named failure.

The other half of the time it really is spelling, and the message names the type (`'str'`), which
tells you which documentation page to open.

`ZeroDivisionError: division by zero`: when it comes from `len(values)`, the message is technically
about arithmetic and actually about an empty list.

### Predict before running · `TypeError` or `ValueError`?

Two failures that beginners routinely swap. Write your prediction for each of the four calls before
you run the cell.

- `int("thirty")` - a string is the right kind for `int()`. Wrong kind, or wrong content?
- `int(None)` - is `None` the right kind?
- `float("3.7.1")` - right kind?
- `len(45)` - can you measure the length of a number?

In [ ]:
# My prediction:
#   int("thirty")  -> ______________
#   int(None)      -> ______________
#   float("3.7.1") -> ______________
#   len(45)        -> ______________

show_error(int, "thirty")
show_error(int, None)
show_error(float, "3.7.1")
show_error(len, 45)

`ValueError`, `TypeError`, `ValueError`, `TypeError`.

**The rule:** `TypeError` = the wrong **kind** of thing, so no possible value of that type would
work. `ValueError` = the right kind, with **content** this function cannot use. `int(None)` can never
work; `int("30")` works fine, so `int("thirty")` is about the content.

The practical consequence arrives in Section 7: values read from a CSV file are always strings, so
`"45" + 5` is a `TypeError` while `int("forty-five")` is a `ValueError`, and the two need different
fixes.

In [ ]:
# 8 · FileNotFoundError -- there is no file at that path (usually: at that WORKING DIRECTORY).
missing = SANDBOX / "does-not-exist.txt"
print("looking for:", missing.resolve())
show_error(open, missing, "r", encoding="utf-8")

# 9 · ModuleNotFoundError -- not installed in THIS environment.
show_error(__import__, "definitely_not_installed")

`FileNotFoundError: [Errno 2] No such file or directory: 'sandbox\does-not-exist.txt'` shows the path
Python actually tried, which is the relative path you gave it - not the absolute one it resolved to.
That is why the cell prints `.resolve()` first: nine times out of ten the file exists and the working
directory is different from what you assumed. Section 5 fixes this for good.

`ModuleNotFoundError: No module named 'definitely_not_installed'` has four causes, in order of
likelihood: it is not installed; it is installed in a *different* Python environment; a typo; or you
named one of your own files `csv.py` or `random.py` and shadowed the real one. The diagnostic for the
second cause is `import sys; print(sys.executable)`, which Section 10 shows.

In [ ]:
# 10 · SyntaxError and IndentationError -- Python could not even read the code.
# These normally stop a cell before anything runs, so here they are handed to compile()
# as text, which lets show_error catch them like any other exception.
show_error(compile, "if minutes = 30:\n    print(minutes)\n", "<demo>", "exec")
show_error(compile, "total = 0\n    total = total + 1\n", "<demo>", "exec")

# 11 · UnboundLocalError -- a local name read before its own assignment.
def count_up(values):
    for value in values:
        total = total + value      # assigning to total makes it LOCAL for the whole function
    return total


show_error(count_up, [1, 2, 3])

A `SyntaxError` is different in kind from the other ten: it happens **before** any of your code runs,
because Python could not parse the file or cell at all. Nothing in the cell executes, so no traceback
frames exist - you get a caret pointing at the offending character instead. `if minutes = 30:` needs
`==`; recent Pythons even say so.

`IndentationError` is a subclass of `SyntaxError`. `unexpected indent` means a line is further right
than its neighbours, which most often happens when tabs and spaces are mixed. Configure your editor
to insert spaces and the whole family disappears.

`UnboundLocalError: cannot access local variable 'total' where it is not associated with a value`
looks impossible until you remember Module 02's scope rule: **assigning to a name anywhere in a
function makes it local everywhere in that function**, including on the line that reads it first.
The fix is `total = 0` before the loop. See
[Module 02 · scope](../02-python-functions-data-structures/index.html#scope).

### Section 2 recap

You have now met all eleven, on purpose:

`NameError` · `TypeError` · `ValueError` · `IndexError` · `KeyError` · `AttributeError` ·
`ZeroDivisionError` · `FileNotFoundError` · `ModuleNotFoundError` · `SyntaxError` /
`IndentationError` · `UnboundLocalError`.

Two habits carry all of this forward:

1. Read the **last line** first, and say the exception's name out loud. If you cannot name it, you
   are guessing.
2. Print the suspect values with `repr` - `print(repr(minutes))` - before changing any code.
   `'45'` and `45` look identical in ordinary `print` output and behave completely differently.

## 3 · try / except / else / finally

Crashing is the right response to a bug and the wrong response to a failure you expected. A file the
user might not have created, a row of data that might be malformed, a number that might not be a
number: these are **expected** failures, and you **handle** them.

To handle an exception is to catch it and continue deliberately. The statement has four clauses:

- `try:` - the risky lines, and as few of them as possible;
- `except SomeError:` - what to do when exactly that goes wrong;
- `else:` - runs only when `try` finished without an exception;
- `finally:` - runs in every single case, on every way out.

In [ ]:
def to_float_or_none(text):
    """Return float(text), or None when text is not a number."""
    try:
        return float(text)
    except ValueError:
        return None


print(to_float_or_none("3.7"))
print(to_float_or_none("-2"))
print(to_float_or_none("oops"))
print(to_float_or_none(" 42 "))        # surrounding whitespace is fine

Three of the four calls returned a number and one returned `None`, and nothing crashed. Note what is
*not* in the `try` block: no printing, no arithmetic, no second conversion. Keep the risky line
alone, so that the `except` clause can only ever be triggered by the failure you were thinking of.

Note also that `float("oops")` was the only thing that could raise here. `to_float_or_none(None)`
would raise a `TypeError`, which this `except ValueError` deliberately does not catch - and that is
correct, because passing `None` to a text-parsing function is a bug, not an expected failure.

### Predict before running · how wide should the net be?

The next cell parses the same bad input three ways: with a bare `except:`, with
`except Exception: pass`, and with `except ValueError:`. One of the three hides a **completely
different** mistake - a misspelt function name.

Predict which lines print, and which of the three versions you would want to be running in a program
whose results you care about.

In [ ]:
# My prediction: the bare version prints ______, and the narrow version prints ______


def parse_bare(text):
    try:
        return flaot(text)             # a typo: flaot, not float
    except:                            # catches absolutely everything
        return None


def parse_silent(text):
    try:
        return flaot(text)             # the same typo
    except Exception:
        pass                           # and now the evidence is gone too


def parse_narrow(text):
    try:
        return flaot(text)             # the same typo again
    except ValueError:
        return None


print("bare  :", parse_bare("3.7"))
print("silent:", parse_silent("3.7"))
show_error(parse_narrow, "3.7")

The first two printed `None` for a perfectly valid input, and told you nothing. The third reported
`NameError: name 'flaot' is not defined` - the actual bug, in the actual place.

**Catch the narrowest exception you can name.** A bare `except:` and `except Exception: pass` are
silent-bug factories: they catch your typos, your `AttributeError`s, and your `KeyError`s along with
the one failure you were prepared for. `pass` is worse than a bare catch, because it also destroys
the message.

If you genuinely want to continue after a failure, say what you caught and why:
`except ValueError as error: print("skipping:", error)`.

In [ ]:
# The exception is an OBJECT. 'as error' gives you a name for it.
def describe_failure(text):
    """Try to parse text, and report what went wrong in detail."""
    try:
        return int(text)
    except (ValueError, TypeError) as error:        # one clause, two exception types
        print("  type   :", type(error).__name__)
        print("  message:", str(error))
        print("  args   :", error.args)
        return None


print("parsing 'thirty':")
describe_failure("thirty")
print("parsing None:")
describe_failure(None)

# Several clauses: the FIRST matching one wins, and the rest are skipped.
def lookup(mapping, key, position):
    try:
        return mapping[key][position]
    except KeyError:
        return "no such learner"
    except IndexError:
        return "no such session"


log = {"maya": [45, 30]}
print(lookup(log, "maya", 1))
print(lookup(log, "ravi", 0))
print(lookup(log, "maya", 9))

`type(error).__name__` gives the exception's name as a string, `str(error)` gives the message, and
`error.args` gives the raw arguments it was constructed with - usually a one-item tuple holding the
message. Those three are how a caught exception becomes a useful report instead of a shrug.

`except (ValueError, TypeError) as error:` catches either, with one body. The parentheses matter: it
is a tuple of types.

With several `except` clauses, Python tries them **in order** and runs the first one that matches.
That ordering is about to become important.

### Exceptions form a family tree

Exception types inherit from each other, exactly as classes do (Section 9 explains the mechanics).
Catching an **ancestor** catches every one of its **descendants**:

```
BaseException
 +-- KeyboardInterrupt          (Ctrl-C)
 +-- SystemExit                 (sys.exit)
 +-- Exception
      +-- ArithmeticError  -->  ZeroDivisionError
      +-- LookupError      -->  IndexError, KeyError
      +-- OSError          -->  FileNotFoundError, FileExistsError, PermissionError
      +-- ValueError, TypeError, NameError --> UnboundLocalError, AttributeError
```

That is why `except Exception:` is so dangerous: it is an ancestor of nearly everything, including
your bugs. And it is why `except BaseException:` or a bare `except:` is worse still - they also
swallow Ctrl-C, so your program can no longer be stopped.

In [ ]:
# Catching an ancestor really does catch the descendants.
print("KeyError is a LookupError? ", issubclass(KeyError, LookupError))
print("FileNotFoundError is an OSError?", issubclass(FileNotFoundError, OSError))
print("ZeroDivisionError is an ArithmeticError?", issubclass(ZeroDivisionError, ArithmeticError))

try:
    {"maya": 135}["bob"]
except LookupError as error:               # never mentions KeyError
    print("caught by 'except LookupError':", type(error).__name__)

# Order matters: a broad clause placed first makes every narrow clause below it dead code.
def wrong_order(mapping, key):
    try:
        return mapping[key]
    except LookupError:
        return "broad clause ran"
    except KeyError:                        # unreachable -- LookupError already matched
        return "narrow clause ran"


print(wrong_order({"maya": 135}, "bob"))

`except LookupError:` caught a `KeyError` without naming it, because `KeyError` is a subclass. This is
useful when you honestly do not care which of a family happened - `except OSError:` for "anything
went wrong with the file" is a reasonable choice.

`wrong_order` returned `"broad clause ran"`. The `except KeyError:` clause below it can never run:
Python takes the first match, and `LookupError` matched first. **Narrow clauses go above broad ones.**
Some editors will warn you about this; Python itself will not.

### `else` and `finally`

- `else:` runs only if `try` completed with no exception. It is where the "and now do the rest of the
  work" lines belong, so that they cannot accidentally trigger the `except` clause.
- `finally:` runs on **every** way out: success, handled exception, unhandled exception, and - the
  part that surprises everyone - past a `return`.

Predict the exact order of printed lines for each of the three calls below before running the cell.

In [ ]:
# My prediction:
#   read_minutes("30")     prints ______________________ and returns ____
#   read_minutes("thirty") prints ______________________ and returns ____
#   read_minutes(None)     prints ______________________ and then ________


def read_minutes(text):
    """Return int(text), or -1 when text is not a number."""
    print("start")
    try:
        value = int(text)
    except ValueError:
        print("except: not a number")
        return -1
    else:
        print("else: parsed", value)
        return value
    finally:
        print("finally: always runs")


print("returned:", read_minutes("30"))
print()
print("returned:", read_minutes("thirty"))
print()
show_error(read_minutes, None)

The order was:

1. `"30"` - `start`, `else: parsed 30`, **`finally: always runs`**, then the value 30 comes back. The
   `return` inside `else` did **not** skip `finally`. Python runs the `finally` block, and only then
   hands over the value it had already computed.
2. `"thirty"` - `start`, `except: not a number`, `finally: always runs`, returns -1.
3. `None` - `int(None)` raises `TypeError`, which `except ValueError` does not catch. `finally` still
   ran, and *then* the exception continued outward, where `show_error` printed it.

That third case is what `finally` is for: closing a file, releasing a lock, restoring a setting -
cleanup that must happen even when you are not handling the failure.

In [ ]:
# Two more facts worth proving, both about return values.

def bad_finally():
    """Never do this: the finally clause REPLACES the value try was about to return."""
    try:
        return "from try"
    finally:
        return "from finally"


check("finally overrides the earlier return", bad_finally(), "from finally")


# Salvaged from the old notebook: fix average() so an empty list is not a crash.
# An 'if' is cleaner than try/except when you can see the problem coming.
def average_or_none(values):
    """Return the mean of values, or None for an empty list."""
    if len(values) == 0:
        return None
    return sum(values) / len(values)


check("average of three scores", average_or_none([90, 85, 77]), 84.0)
check("average of []", average_or_none([]), None)

`bad_finally()` returned `"from finally"`. A `return` inside `finally` throws away the value the
function was already committed to - and it also swallows any exception on its way out. Treat "never
`return` from `finally`" as an absolute rule.

`average_or_none` shows the other half of the judgement. `try`/`except` is for failures you cannot
check for in advance; an emptiness test you can see coming belongs in an `if`. Python's guidance is
"ask forgiveness, not permission" for genuine races and unpredictable inputs, and a plain `if` for
conditions you can simply look at.

Returning `None` here is still not ideal - Section 4 shows why raising is usually the better answer.

**Section 3 recap:** narrow catches, ordered narrow-to-broad · `as error` for the details ·
`else` for the rest of the work · `finally` for cleanup, always, even past a `return` · never
`except Exception: pass`.

## 4 · Raising, chaining, and asserting

To **raise** is to start an exception yourself. It is how a function refuses bad input instead of
quietly producing nonsense, and it is the other half of the skill you just practised: some code
raises, other code handles.

Validate at the **top** of the function, before any real work happens.

In [ ]:
def minutes_to_hours(minutes):
    """Return minutes as hours. Raise ValueError for a negative or absurd value."""
    if not isinstance(minutes, (int, float)):
        raise TypeError("minutes must be a number, got " + type(minutes).__name__)
    if minutes < 0:
        raise ValueError("minutes cannot be negative: " + repr(minutes))
    if minutes > 1440:
        raise ValueError("more than a day of study is not credible: " + repr(minutes))
    return minutes / 60


print(minutes_to_hours(90))
show_error(minutes_to_hours, -5)
show_error(minutes_to_hours, 5000)
show_error(minutes_to_hours, "45")

Three guard clauses, each raising the exception whose meaning matches: `TypeError` for the wrong
kind, `ValueError` twice for the right kind with impossible content. The messages include the
offending value with `repr`, which is what makes them useful in a traceback six frames away from
here.

Note `1440`: the rule "more than a day is not credible" is a **domain** decision, not a Python one.
Writing it down as a raise is how a rule stops being folklore.

### Predict before running · raise, or return `None`?

Two versions of the same loader. One raises on a bad row; the other returns `None` and lets the
caller carry on. Predict which line number the eventual crash is reported on in each case, and which
one you would rather debug at 11pm.

In [ ]:
# My prediction: the returning version crashes ______________________________


def total_raising(rows):
    """Sum the minutes column, raising on a bad row."""
    total = 0
    for row in rows:
        if not row.isdigit():
            raise ValueError("row is not a number: " + repr(row))
        total = total + int(row)
    return total


def total_returning(rows):
    """Sum the minutes column, returning None on a bad row."""
    total = 0
    for row in rows:
        if not row.isdigit():
            return None                    # a 'sentinel' value: looks harmless, is not
        total = total + int(row)
    return total


good = ["45", "30", "60"]
bad = ["45", "thirty", "60"]

print("raising, good rows :", total_raising(good))
show_error(total_raising, bad)

print("returning, good rows:", total_returning(good))
answer = total_returning(bad)
print("returning, bad rows :", answer)
show_error(minutes_to_hours, answer)       # the crash arrives HERE, far from the cause

The raising version failed at the moment it met the bad row, and its message named the row.

The returning version succeeded, handed back `None`, and the crash arrived later, in a function that
had nothing to do with parsing: `TypeError: minutes must be a number, got NoneType`. A **sentinel**
return value does not prevent a crash - it relocates it somewhere confusing and deletes the evidence
on the way.

**Raise on bad input.** Return `None` only when "nothing" is a genuinely valid answer that every
caller expects, such as `dict.get`.

In [ ]:
# Your own exception classes. These three are the project's, and the names are fixed.
class StudyLogError(Exception):
    """Base class for every failure this project raises on purpose."""


class InvalidEntryError(StudyLogError, ValueError):
    """A row we refuse to accept."""


class NotFittedError(StudyLogError):
    """predict() was called before fit()."""


print("InvalidEntryError is a StudyLogError?", issubclass(InvalidEntryError, StudyLogError))
print("InvalidEntryError is a ValueError?   ", issubclass(InvalidEntryError, ValueError))
print("NotFittedError is a ValueError?      ", issubclass(NotFittedError, ValueError))


def parse_row(row_number, text):
    """Return int(text), or raise InvalidEntryError naming the row."""
    if not text.strip().isdigit():
        raise InvalidEntryError(f"row {row_number}: minutes is not a number: {text!r}")
    return int(text)


check("a good row parses", parse_row(2, "45"), 45)
check_raises("a bad row raises InvalidEntryError", InvalidEntryError, parse_row, 3, "thirty")
check_raises("and the same object IS a ValueError", ValueError, parse_row, 3, "thirty")
check_raises("and it IS a StudyLogError", StudyLogError, parse_row, 3, "thirty")

Three classes and three `pass`-free bodies: a docstring is enough of a body, so no `pass` is needed.

`class InvalidEntryError(StudyLogError, ValueError):` lists **two** parents, and that is a kindness
to whoever calls your code. A caller who knows nothing about your project can write
`except ValueError:` and it works. A caller who wants everything this project raises can write
`except StudyLogError:` and that works too. One exception, two honest descriptions of itself.

`NotFittedError` is deliberately *not* a `ValueError`: calling `predict()` before `fit()` is not a bad
value, it is a wrong order of operations.

Section 9 explains the inheritance machinery. For now the pattern is enough: a class, a base class in
parentheses, and a docstring.

In [ ]:
# Chaining: catch a low-level failure, add context, and keep the original attached.
def load_minutes(row_number, text):
    """Convert text to int, reporting the row. Keeps the original error as the cause."""
    try:
        return int(text)
    except ValueError as error:
        raise InvalidEntryError(f"row {row_number}: bad minutes {text!r}") from error


try:
    load_minutes(3, "thirty")
except InvalidEntryError as error:
    print("what escaped :", type(error).__name__, "-", error)
    print("its cause    :", type(error.__cause__).__name__, "-", error.__cause__)
    print("__cause__ set?", error.__cause__ is not None)

# Without 'from', the original is still attached -- as __context__, and the traceback
# says "During handling of the above exception, another exception occurred".
def load_minutes_unchained(row_number, text):
    try:
        return int(text)
    except ValueError:
        raise InvalidEntryError(f"row {row_number}: bad minutes {text!r}")


try:
    load_minutes_unchained(3, "thirty")
except InvalidEntryError as error:
    print()
    print("with no 'from': __cause__ is", error.__cause__, "but __context__ is",
          type(error.__context__).__name__)

`raise NewError(...) from error` sets `__cause__`, and that is what produces the sentence you read in
Section 1: *The above exception was the direct cause of the following exception.* You get a message
in your own vocabulary (`row 3: bad minutes 'thirty'`) without throwing away the machine-level
detail (`invalid literal for int()`).

Leaving out `from` still records the original, as `__context__`, and Python prints *During handling of
the above exception, another exception occurred* - which reads like an accident. Use `from error`
whenever the replacement is deliberate. Use `raise` with no arguments to re-raise the exception you
are currently handling, unchanged, after logging or cleaning up.

In [ ]:
# assert: for a state that should be IMPOSSIBLE, never for input a user or a file controls.
def blend(weight_a, weight_b, value_a, value_b):
    """Combine two values by weight. The weights must already sum to 1."""
    assert abs(weight_a + weight_b - 1.0) < 1e-9, "weights must sum to 1"
    return weight_a * value_a + weight_b * value_b


print(blend(0.25, 0.75, 40, 80))
show_error(blend, 0.5, 0.9, 40, 80)         # AssertionError -- a programmer mistake, not user input

# EAFP -- 'easier to ask forgiveness than permission' -- versus LBYL, 'look before you leap'.
target = SANDBOX / "eafp.txt"
with open(target, "w", encoding="utf-8") as handle:
    handle.write("45\n")

# LBYL: two separate moments, and the file could vanish between them.
if target.exists():
    with open(target, "r", encoding="utf-8") as handle:
        print("LBYL read:", handle.read().strip())

# EAFP: one moment, and the failure is handled where it happens.
try:
    with open(target, "r", encoding="utf-8") as handle:
        print("EAFP read:", handle.read().strip())
except FileNotFoundError:
    print("EAFP: no such file")

# And the debugging habit that saves the most time: repr, not print.
value_from_a_file = "30\n"
print("print says :", value_from_a_file, "| looks like a number")
print("repr says  :", repr(value_from_a_file), "| a string, with a newline inside")
print("repr of a space-padded value:", repr(" 45 "))

Three ideas, one cell.

`assert condition, "message"` raises `AssertionError` when the condition is false. It documents an
assumption *inside* your own code. It is the wrong tool for validating input, for two reasons: the
message is aimed at you rather than the user, and running Python with `-O` **removes every assert
statement**, so validation written as an assert simply disappears in that mode. Use `raise` for
anything a user, a file, or a network can cause; use `assert` for "this can only be wrong if I made a
mistake".

**EAFP vs LBYL:** checking `exists()` and then opening is two operations with a gap in between, and
in that gap the file can be deleted. Wrapping the `open` in `try`/`except FileNotFoundError` has no
gap. Python's habit is EAFP for anything involving files, dictionaries, or conversion.

**`repr`, not `print`:** `print` showed `30` followed by a line break you cannot see; `repr` showed
`'30\n'` - a string, with a newline in it. Every "but it looks right!" bug in Sections 6 and 7 is
found this way.

### A debugging routine that always works

1. **Read the last line.** Name the exception out loud.
2. **Find your own newest frame.** Scan upward past library frames to the last file you wrote.
3. **Print the inputs with `repr`.** `print(repr(row), repr(minutes))` before anything else.
4. **Shrink it.** Delete everything not needed to reproduce the failure. A five-line reproduction
   usually explains itself.
5. **Bisect.** Comment out half. Still broken? The bug is in the half that ran. Repeat.

Then, if it is still standing: explain the code out loud, line by line, to a rubber duck. Stating the
expectation is what exposes the gap.

**Section 4 recap:** validate at the top and `raise` · a sentinel `None` relocates the crash ·
custom exception classes are three lines and worth it · `from error` for a deliberate replacement ·
`assert` for impossible states only · `repr` before you change any code.

## 5 · Paths and the working directory

`FileNotFoundError` on a file you can see in the folder is the most common blocker in this whole
module, and it is almost never about the file. It is about **where "here" is**.

- The **working directory** is the folder your program considers "here".
- A **relative path** such as `"sample_log.csv"` or `"data/log.csv"` is resolved against the working
  directory - **not** against the file you are editing.
- An **absolute path** starts from the drive or root and needs no context.

The rule: **a relative path means nothing until you know the working directory.**

In [ ]:
from pathlib import Path

print("working directory:", Path.cwd())
print()

# Building paths: the / operator joins them, correctly, on every operating system.
data_dir = SANDBOX / "data"
log_path = data_dir / "log.csv"

print("data_dir     :", data_dir)
print("log_path     :", log_path)
print("resolved     :", log_path.resolve())
print("is absolute? ", log_path.is_absolute(), "->", log_path.resolve().is_absolute())

# A notebook has no __file__. A script does. This is why the same code can work in one and
# fail in the other.
print()
print("does this notebook have __file__?", "__file__" in globals())
print("in a SCRIPT you would write: Path(__file__).parent / 'sample_log.csv'")

`SANDBOX / "data"` did not touch the disk: a `Path` is just a value describing a location, and you can
build and print one for a file that does not exist. `/` between paths is `pathlib`'s join operator,
and it inserts the right separator for the operating system you are on - which is why nothing in this
notebook contains a hard-coded `\` or `/` inside a filename.

`__file__` is the running file's own path. **A notebook does not have one**, because there is no
single file being executed - which is exactly why `Path(__file__).parent / "sample_log.csv"` is the
right idiom in `study_tracker.py` and unavailable here. In a notebook, use `Path.cwd()` or a path
relative to it, and check with `.resolve()` when something surprises you.

In [ ]:
# The parts of a path, and creating folders.
data_dir.mkdir(parents=True, exist_ok=True)      # parents=True also creates missing parents
log_path.write_text("name,minutes\nmaya,45\n", encoding="utf-8")

print("full path :", log_path)
print("name      :", log_path.name)              # log.csv
print("stem      :", log_path.stem)              # log
print("suffix    :", log_path.suffix)            # .csv
print("parent    :", log_path.parent)
print("parent's parent:", log_path.parent.parent)
print("exists?   ", log_path.exists(), " is a file?", log_path.is_file())
print("size      :", log_path.stat().st_size, "bytes")

# Going up with '..' works, but resolve() is what makes it readable.
print("up and over:", (log_path.parent / ".." / "data" / "log.csv").resolve())

Six attributes cover almost every path question: `.name`, `.stem`, `.suffix`, `.parent`, `.exists()`,
and `.resolve()`. None of them needs string slicing, and none of them breaks on Windows.

`.mkdir(parents=True, exist_ok=True)` is the safe pair of options: create any missing parent folders,
and do not complain if the folder is already there. **Writing to a file in a folder that does not
exist raises `FileNotFoundError`, not "folder created"** - so create the folder first. That is what
Section 0's `SANDBOX.mkdir(exist_ok=True)` was for.

`..` means "the folder above". It works, but a path with `..` in the middle is hard to read; call
`.resolve()` and print it when you are unsure.

### Predict before running · the backslash trap

This course runs on Windows, where paths are shown with backslashes. But `\` in a Python string starts
an **escape sequence**: `\n` is a newline, `\t` is a tab. So a Windows path pasted straight into
quotes can silently change meaning.

Predict how many characters `"data\names.txt"` contains, and what its `repr` will look like.

In [ ]:
# My prediction: len is ______ and the repr shows ____________________

broken = "data\names.txt"                 # \n became a NEWLINE
raw_ok = r"data\names.txt"                # r"..." keeps the backslash literal
slash_ok = "data/names.txt"               # forward slashes work on Windows too
best = SANDBOX / "data" / "names.txt"     # pathlib: no separator typed at all

print("broken :", repr(broken), "len", len(broken))
print("raw_ok :", repr(raw_ok), "len", len(raw_ok))
print("slash  :", repr(slash_ok))
print("best   :", best)
print()
print("printed without repr, the broken one hides its own bug:")
print(broken)

`"data\names.txt"` is 13 characters, not 14, and its `repr` is `'data\names.txt'` - the `\n` is one
character, a newline. Printed plainly it looks like two lines of nonsense; printed with `repr` the bug
is obvious. This is the `repr` habit from Section 4 earning its keep.

Three fixes, in increasing order of preference: a raw string `r"data\names.txt"`, forward slashes
(Windows accepts them everywhere in Python), and best of all **no separator in a string at all** -
build the path with `pathlib` and the `/` operator.

Two more traps worth naming: a hard-coded `C:\Users\you\Desktop\...` path works on exactly one
computer, and `..` chains break as soon as someone moves the file. Both are solved by building from
`Path.cwd()` in a notebook or `Path(__file__).parent` in a script.

In [ ]:
# Listing what is actually there -- the fastest cure for a FileNotFoundError.
(data_dir / "extra.csv").write_text("name,minutes\nada,90\n", encoding="utf-8")
(data_dir / "notes.txt").write_text("not a csv\n", encoding="utf-8")

print("everything in", data_dir.resolve(), "->")
for item in sorted(data_dir.iterdir()):
    print("   ", item.name, "|", item.stat().st_size, "bytes")

print()
print("only the CSV files:", sorted(p.name for p in data_dir.glob("*.csv")))
print("relative to sandbox:", sorted(str(p.relative_to(SANDBOX)) for p in data_dir.glob("*.csv")))

`.iterdir()` yields every entry in a folder; `.glob("*.csv")` yields the ones matching a pattern. When
a path does not work, print the resolved folder and list it: you will see the typo, the wrong
extension, or the wrong folder immediately.

**Section 5 recap.** In a notebook, paths are relative to `Path.cwd()`. In a script, build them from
`Path(__file__).parent` so the script runs from any directory - which is exactly what
`study_tracker.py` in this folder does. `os.path` is the older API for the same job, all strings; you
will see it in other people's code, but prefer `pathlib`.

## 6 · Files: modes, encoding, round trips

Opening a file needs three decisions, and one of them can destroy data:

| Mode | Meaning | If the file is missing | If it exists |
|---|---|---|---|
| `"r"` | read | `FileNotFoundError` | reads it |
| `"w"` | write | creates it | **empties it immediately** |
| `"a"` | append | creates it | adds at the end |
| `"x"` | create | creates it | `FileExistsError` |

And two rules with no exceptions in this course: always use `with`, and always pass
`encoding="utf-8"`.

In [ ]:
target = SANDBOX / "scores.txt"

# The idiom to memorise. 'with' closes the file on the way out, even through an exception.
with open(target, "w", encoding="utf-8") as handle:
    handle.write("90\n")
    handle.write("85\n")
    handle.write("77\n")

with open(target, "r", encoding="utf-8") as handle:
    contents = handle.read()

print(repr(contents))
print("closed automatically?", handle.closed)

`with open(...) as handle:` opens the file, gives you the object, and **closes it when the block
ends** - on the normal path, on a `return`, and on an exception. The bare alternative,
`handle = open(...)` followed by `handle.close()`, is wrong the first time anything raises in between,
because the `close()` line never runs.

An object usable in a `with` block is called a **context manager**. You will use dozens of them and
rarely write one; the mechanics are two dunder methods, `__enter__` and `__exit__`, which Section 8
will make sense of.

Note `repr(contents)`: `'90\n85\n77\n'`. Three lines, and the trailing newline on the last one is
visible.

### Predict before running · what `"w"` does, and when

You have a file containing three scores. You open it with mode `"w"` and then, before writing
anything at all, you check its size on disk.

Predict that size.

In [ ]:
# My prediction: the size the instant "w" opens the file is ______ bytes

print("size before   :", target.stat().st_size, "bytes")

# Deliberately NOT using 'with' here, so that we can look at the file mid-way.
handle = open(target, "w", encoding="utf-8")
print("size on open  :", target.stat().st_size, "bytes   <-- nothing has been written yet")
handle.write("60\n")
handle.close()
print("size after    :", target.stat().st_size, "bytes")
print("contents now  :", repr(target.read_text(encoding="utf-8")))

Zero. **`"w"` empties the file the moment it is opened**, before a single byte of yours arrives. If
your program crashes on the line after the `open`, the old contents are already gone.

This is the most expensive beginner mistake with files, and it has nothing to do with `write`. The
lesson: choose the mode from your **intent**.

- I am replacing the whole file, and losing the old contents is fine -> `"w"`.
- I am adding to what is there -> `"a"`.
- I must not overwrite an existing file -> `"x"`, and handle `FileExistsError`.
- I am only reading -> `"r"`.

(The bare `open` above is the only one in this notebook, and it exists purely to freeze that instant.
Everywhere else, use `with`.)

In [ ]:
# "a" appends and creates; "x" refuses to clobber; "r" refuses to write.
with open(target, "a", encoding="utf-8") as handle:
    handle.write("55\n")
print("after append :", repr(target.read_text(encoding="utf-8")))

fresh = SANDBOX / "brand_new.txt"
if fresh.exists():
    fresh.unlink()                       # start from a known state so this cell can rerun

with open(fresh, "x", encoding="utf-8") as handle:
    handle.write("created by x\n")
print("x created it :", repr(fresh.read_text(encoding="utf-8")))

show_error(open, fresh, "x", encoding="utf-8")          # FileExistsError -- it already exists
show_error(open, SANDBOX / "no-such-file.txt", "r", encoding="utf-8")   # FileNotFoundError


def write_to_a_read_handle(path):
    """Open for reading and try to write, on purpose."""
    with open(path, "r", encoding="utf-8") as handle:
        return handle.write("nope")


show_error(write_to_a_read_handle, target)              # io.UnsupportedOperation: not writable

Four modes, four distinct behaviours, and three of them announce a problem instead of guessing:

- `"a"` preserved `60` and added `55`. It also creates the file when it is missing, which makes it the
  safe default for a log.
- `"x"` created the file the first time and raised `FileExistsError` the second. That is the mode to
  use when overwriting would be a disaster.
- `"r"` raised `FileNotFoundError` for a missing file, and `io.UnsupportedOperation: not writable`
  when asked to write. A read handle is genuinely read-only.

`path.unlink()` deletes a file. It is used here only so the cell can be run twice.

In [ ]:
# Four ways to read, and when each is right.
with open(target, "w", encoding="utf-8") as handle:
    handle.write("90\n85\n77\n")

with open(target, "r", encoding="utf-8") as handle:
    everything = handle.read()            # the whole file as ONE string
print("read()      :", repr(everything))

with open(target, "r", encoding="utf-8") as handle:
    first = handle.readline()             # one line, including its newline
    second = handle.readline()
print("readline()  :", repr(first), repr(second))

with open(target, "r", encoding="utf-8") as handle:
    all_lines = handle.readlines()        # a LIST of lines, all in memory
print("readlines() :", all_lines)

with open(target, "r", encoding="utf-8") as handle:
    for number, line in enumerate(handle, start=1):    # one line at a time -- the cheapest
        print(f"  line {number}: {line!r}")

| Call | Gives you | Use it when |
|---|---|---|
| `.read()` | one string | the file is small and you want it whole |
| `.readline()` | the next line | you are reading a header, then the rest |
| `.readlines()` | a list of lines | you need indexing or a length, and the file is small |
| iteration | one line at a time | **the default** - works on a file of any size |

Iteration is the one to reach for. It never holds more than one line in memory, which matters the
first time you meet a data file bigger than your RAM. `enumerate(handle, start=1)` gives you line
numbers for free, and a line number in an error message is worth a great deal - Section 7's loader
uses exactly this.

In [ ]:
# Every line carries its newline. .strip() is not optional.
with open(target, "r", encoding="utf-8") as handle:
    raw = [line for line in handle]

print("raw          :", raw)
print("stripped     :", [line.strip() for line in raw])
show_error(int, raw[0])                 # does int() mind the newline? predict, then read
print("int of stripped:", int(raw[0].strip()))

# Writing: neither .write nor .writelines adds a newline for you.
joined = SANDBOX / "joined.txt"
with open(joined, "w", encoding="utf-8") as handle:
    handle.write("a")
    handle.write("b")
    handle.writelines(["c", "d"])       # note: no separators added
print("no newlines added:", repr(joined.read_text(encoding="utf-8")))

with open(joined, "w", encoding="utf-8") as handle:
    handle.writelines(line + "\n" for line in ["a", "b", "c"])
print("newlines supplied:", repr(joined.read_text(encoding="utf-8")))

`int("90\n")` actually works - `int()` tolerates surrounding whitespace - but `"90\n" == "90"` is
`False`, and a dictionary keyed on unstripped names will have `'maya'` and `'maya\n'` as two different
learners. **Strip every line you read.**

Neither `.write()` nor `.writelines()` adds a newline. `.writelines` is a slightly misleading name: it
writes the strings you give it, unchanged, with nothing in between. Supply the `"\n"` yourself, as the
generator expression in the last block does.

In [ ]:
# Text is bytes. An encoding is the decoder ring.
cafe = SANDBOX / "cafe.txt"
with open(cafe, "w", encoding="utf-8") as handle:
    handle.write("cafe\ncafé\nnaïve\n")

with open(cafe, "r", encoding="utf-8") as handle:
    print("utf-8 read :", repr(handle.read()))

# Symptom 1: a crash. The bytes cannot be decoded with the encoding you named.
show_error(cafe.read_text, encoding="ascii")

# Symptom 2: MOJIBAKE -- no crash, wrong characters. This is the dangerous one.
print("latin-1 read:", repr(cafe.read_text(encoding="latin-1")))
print("            ", cafe.read_text(encoding="latin-1").replace("\n", " | "))

The same bytes, read three ways: correct, a crash, and silent corruption.

- `encoding="utf-8"` gave back exactly what was written.
- `encoding="ascii"` raised `UnicodeDecodeError: 'ascii' codec can't decode byte 0xc3 ...`. A crash is
  the *good* outcome: it tells you the encoding is wrong.
- `encoding="latin-1"` produced `cafÃ©` and `naÃ¯ve`. No error, wrong text. That is **mojibake**, and
  it is dangerous precisely because nothing complains.

**Always pass `encoding="utf-8"`, on the way in and on the way out.** If you omit it, Python uses the
system default, which on Windows has historically been `cp1252` - so a file that works on a
colleague's Mac fails on your laptop, and the failure is sometimes silent. One extra argument removes
an entire category of bug. (`encoding="utf-8-sig"` handles a file exported from Excel that starts with
a byte-order mark; use it only when `utf-8` gives you a stray `\ufeff` on the first line.)

In [ ]:
# The skip-don't-crash loader: Sections 3 and 4, applied to a real messy file.
messy = SANDBOX / "messy_scores.txt"
with open(messy, "w", encoding="utf-8") as handle:
    handle.write("90\n85\noops\n\n77\n")


def load_scores_reporting(path):
    """Return every parseable line as a float, reporting each line it had to skip."""
    scores = []
    skipped = []
    with open(path, "r", encoding="utf-8") as handle:
        for number, line in enumerate(handle, start=1):
            text = line.strip()
            if not text:
                continue                        # blank lines are not errors
            try:
                scores.append(float(text))
            except ValueError as error:
                skipped.append(number)
                print(f"  skipping line {number}: {error}")
    return scores, skipped


values, skipped_lines = load_scores_reporting(messy)
check("parsed scores", values, [90.0, 85.0, 77.0])
check("skipped lines", skipped_lines, [3])

This is the pattern every loader in the rest of the course uses:

1. iterate, with a line number;
2. skip what is genuinely empty;
3. put **only** the conversion inside `try`;
4. catch the narrowest exception, report it *with its line number*, and carry on;
5. return the good data, and say what was dropped.

A loader that crashes on one bad row out of ten thousand is unusable. A loader that silently drops
rows is worse. Reporting the line number is what makes it neither.

**Section 6 recap:** `with` always · `encoding="utf-8"` always · `"w"` truncates on `open` ·
iterate rather than `.readlines()` · `.strip()` every line · `.write` adds no newline · mojibake is
the silent failure to fear.

## 7 · CSV and JSON

**CSV** - comma-separated values - is the format your data will actually arrive in: one header row,
then one row per record. **JSON** - JavaScript Object Notation - is the format for nested data and
configuration, and it keeps types.

First, a file to work with. The next cell writes the module's `sample_log.csv` into `sandbox/`, so
that everything below works even if the copy next to the notebook has been moved or edited.

In [ ]:
SAMPLE_LOG_TEXT = (
    "name,date,minutes,topic\n"
    "maya,2026-03-02,45,functions\n"
    "maya,2026-03-03,30,lists\n"
    "maya,2026-03-04,60,dicts\n"
    "ravi,2026-03-02,20,functions\n"
    "ravi,2026-03-03,25,lists\n"
    "ravi,2026-03-05,15,sets\n"
    "ada,2026-03-02,90,comprehensions\n"
    "ada,2026-03-04,75,aliasing\n"
)

SAMPLE_LOG = SANDBOX / "sample_log.csv"
with open(SAMPLE_LOG, "w", encoding="utf-8", newline="") as handle:
    handle.write(SAMPLE_LOG_TEXT)

print("wrote", SAMPLE_LOG.resolve(), "-", SAMPLE_LOG.stat().st_size, "bytes")
print("the copy shipped beside the notebook exists?", Path("sample_log.csv").exists())
print()

# By hand first: strip the newline, split on the comma.
with open(SAMPLE_LOG, "r", encoding="utf-8") as handle:
    header = handle.readline().strip().split(",")
    print("header:", header)
    for line in handle:
        fields = line.strip().split(",")
        print("row   :", fields)

`line.strip().split(",")` is a complete CSV reader for clean data, and worth knowing: it shows there
is no magic in the format. A header line names the columns, and each following line has the same
number of fields in the same order.

Note `newline=""` on the write. Section 6 said always pass `encoding`; for CSV files there is a second
argument, and the next cells show what it prevents.

The by-hand approach has one fatal weakness.

### Predict before running · a comma inside a field

A learner studies "functions, lists and dicts" in one session. In a CSV, that topic is written
`"functions, lists and dicts"` - quoted, because it contains the separator.

Predict how many fields `line.strip().split(",")` finds in that row, and what the `minutes` field
ends up being.

In [ ]:
# My prediction: split(",") finds ______ fields, and minutes is ______________

quoted = SANDBOX / "quoted.csv"
with open(quoted, "w", encoding="utf-8", newline="") as handle:
    handle.write("name,topic,minutes\n")
    handle.write('maya,"functions, lists and dicts",45\n')

print("the file itself:")
print(quoted.read_text(encoding="utf-8"))

with open(quoted, "r", encoding="utf-8") as handle:
    handle.readline()
    hand_rolled = handle.readline().strip().split(",")
print("by hand    :", hand_rolled, "->", len(hand_rolled), "fields")

import csv

with open(quoted, "r", encoding="utf-8", newline="") as handle:
    reader = csv.reader(handle)
    next(reader)                                # skip the header row
    proper = next(reader)
print("csv.reader :", proper, "->", len(proper), "fields")

By hand: four fields, and `minutes` came out as `' lists and dicts'`. The `csv` module: three fields,
and `minutes` is `'45'`.

The quoting rules of real CSV are not hard, but they are not nothing: quoted fields, doubled quotes
inside quoted fields, embedded newlines. `csv` implements all of them. **Use the module.** Writing
your own is fine for a file you generated yourself five minutes ago and nothing else.

In [ ]:
# csv.DictReader: each row arrives as a dict keyed by the header.
with open(SAMPLE_LOG, "r", encoding="utf-8", newline="") as handle:
    reader = csv.DictReader(handle)
    print("field names:", reader.fieldnames)
    rows = list(reader)

print("rows read  :", len(rows))
print("first row  :", rows[0])
print()
print("the minutes field:", repr(rows[0]["minutes"]), "of type", type(rows[0]["minutes"]).__name__)


def add_five(row):
    """Add 5 to a row's minutes without converting first, on purpose."""
    return row["minutes"] + 5


show_error(add_five, rows[0])
print("with a conversion:", int(rows[0]["minutes"]) + 5)

`DictReader` reads the header row for you and hands each following row back as a dict, so you write
`row["minutes"]` instead of `fields[2]` and stop counting columns.

And then the trap that causes more downstream `TypeError`s than anything else in the course:
**everything that comes out of a CSV file is a string.** `rows[0]["minutes"]` is `'45'`, not `45`, so
`row["minutes"] + 5` is `TypeError: can only concatenate str (not "int") to str`.

Convert at the **boundary** - the moment a row arrives - not scattered through the rest of your code.
And when the conversion fails, that is exactly where `InvalidEntryError` with a row number belongs.
Compare `'45'` and `45` with `repr` whenever a number "isn't behaving like a number".

In [ ]:
# Convert at the boundary, skip bad rows, and report them with a row number.
def load_rows(path):
    """Return (entries, skips) for a study-log CSV. Converts minutes at the boundary."""
    entries = []
    skips = []
    with open(path, "r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        for row_number, row in enumerate(reader, start=2):     # start=2: line 1 is the header
            try:
                minutes = int(row["minutes"])
            except ValueError as error:
                message = f"skipping row {row_number}: minutes must be a whole number, got {row['minutes']!r}"
                skips.append(message)
                print(message)
                continue
            entries.append({"name": row["name"], "date": row["date"],
                            "minutes": minutes, "topic": row["topic"]})
    return entries, skips


entries, skips = load_rows(SAMPLE_LOG)
check("all eight rows loaded", len(entries), 8)
check("nothing skipped", skips, [])

# The same loader on a file with one bad row.
one_bad = SANDBOX / "one_bad_row.csv"
with open(one_bad, "w", encoding="utf-8", newline="") as handle:
    handle.write(SAMPLE_LOG_TEXT.replace("ravi,2026-03-02,20,functions",
                                        "ravi,2026-03-02,twenty,functions"))

partial, partial_skips = load_rows(one_bad)
check("seven good rows survived", len(partial), 7)
check("one skip, with its row number", partial_skips,
      ["skipping row 5: minutes must be a whole number, got 'twenty'"])

# Totals, to be reused in Section 12.
totals = {}
for entry in entries:
    totals[entry["name"]] = totals.get(entry["name"], 0) + entry["minutes"]
print()
print("totals:", totals)
check("maya's total", totals["maya"], 135)
check("ravi's total", totals["ravi"], 60)
check("ada's total", totals["ada"], 165)
check("class total", sum(totals.values()), 360)

`enumerate(reader, start=2)` numbers the rows so that the number matches the **line number in the
file**: the header is line 1, so the first data row is line 2. When a user opens the file to fix your
reported row, that number has to be the one their editor shows.

One malformed row cost one row, not the run. Seven of eight entries survived, and the skip was
reported precisely enough to fix. This is the behaviour the mini-project's requirements demand.

The totals - maya 135, ravi 60, ada 165, class 360 - are the numbers the rest of this notebook and
the lesson page both use.

In [ ]:
# Writing a CSV: DictWriter plus writeheader, and newline="" every time.
summary_csv = SANDBOX / "summary.csv"
rows_out = [{"name": name, "total": total, "sessions": 0} for name, total in sorted(totals.items())]
for row in rows_out:
    row["sessions"] = len([e for e in entries if e["name"] == row["name"]])

with open(summary_csv, "w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["name", "total", "sessions"])
    writer.writeheader()
    writer.writerows(rows_out)

print("the file, byte for byte:")
print(repr(summary_csv.read_text(encoding="utf-8")))
print()
print(summary_csv.read_text(encoding="utf-8"))

with open(summary_csv, "r", encoding="utf-8", newline="") as handle:
    back = list(csv.DictReader(handle))
check("round trip kept every row", len(back), 3)
check("but the numbers came back as text", back[0]["total"], "165")
check("ada's average", int(back[0]["total"]) / int(back[0]["sessions"]), 82.5)

`csv.DictWriter` needs `fieldnames` - it will not guess the column order - and `writeheader()` is a
separate call you must remember.

**`newline=""` is not optional.** A CSV writer emits `\r\n` itself; if you let Python's text layer
also translate `\n` to `\r\n`, you get `\r\r\n` and a blank line between every row on Windows. The
`repr` above shows the real bytes, so you can see that there is exactly one `\r\n` per row.

Note the round trip: `165` went out as an integer and came back as the string `'165'`. CSV has no
types at all. That is the single reason JSON exists in this section.

### Predict before running · what survives a JSON round trip?

JSON has six types: object (a dict), array (a list), string, number, boolean, and null. Python has
more than that, so some values cannot make the trip unchanged.

Predict, for each of these, what comes back after `json.dump` then `json.load`:

- a tuple `(3, 1)`;
- a dict with **integer** keys, `{1: "one", 2: "two"}`;
- a set `{"lists", "dicts"}`;
- a float `45.0` and an int `45`.

In [ ]:
# My prediction:
#   (3, 1)            -> ______________
#   {1: "one"}        -> ______________
#   {"lists","dicts"} -> ______________
#   45.0 and 45       -> ______________

import json

payload = {
    "learner": "maya",
    "point": (3, 1),
    "counts": {1: "one", 2: "two"},
    "average": 45.0,
    "sessions": 3,
    "active": True,
    "notes": None,
}

payload_path = SANDBOX / "payload.json"
with open(payload_path, "w", encoding="utf-8") as handle:
    json.dump(payload, handle, indent=2)          # indent=2 for a file a human will read

with open(payload_path, "r", encoding="utf-8") as handle:
    restored = json.load(handle)

print(payload_path.read_text(encoding="utf-8"))
print("restored:", restored)
print()
check("a tuple came back as a list", restored["point"], [3, 1])
check("integer keys came back as strings", sorted(restored["counts"]), ["1", "2"])
check("floats survive", restored["average"], 45.0)
check("ints survive as ints", restored["sessions"], 3)
check("True survives", restored["active"], True)
check("None survives", restored["notes"], None)

show_error(json.dumps, {"topics": {"lists", "dicts"}})     # a set is not JSON at all

Three things changed, and two of them change silently:

- **A tuple becomes a list.** JSON has one sequence type. `(3, 1)` goes out as `[3, 1]` and comes back
  as a list, so `restored["point"][0] = 9` now works when it used to raise.
- **Dict keys become strings.** JSON object keys are always text, so `1` becomes `"1"`. Code that does
  `counts[1]` after a round trip gets a `KeyError`.
- **A set raises `TypeError: Object of type set is not JSON serializable`** - a loud, honest failure.
  Convert to a sorted list before saving.

Everything else survives: strings, ints, floats, `True`/`False` (JSON `true`/`false`), and `None`
(JSON `null`). `json.dumps` and `json.loads` - with the `s` - do the same work with a **string**
instead of a file, which is what you want when the JSON arrives from a web response.

### CSV or JSON?

| | CSV | JSON |
|---|---|---|
| shape | one flat table | nested, any depth |
| types | everything is text | int, float, bool, null, str survive |
| size | compact | larger, especially with `indent` |
| tools | opens in any spreadsheet | reads in any language, awkward in Excel |
| good for | rows of measurements | configuration, mixed or nested records |

Rule of thumb: **rows of the same shape -> CSV. Anything nested or typed -> JSON.**

### Why pandas exists

Grouping the log by learner took a loop, a `.get()` with a fallback, a second pass for the counts, and
a comprehension - about a dozen lines, all of which you can now write and debug. In Module 08 the same
result is:

```python
df.groupby("name").minutes.mean()
```

One line, and the types are read from the file rather than converted by hand. That is what pandas is
*for*, and it is also why this section exists: pandas is a convenience over exactly the operations you
just performed, and when it does something surprising you will need this section to work out what.

**Section 7 recap:** use the `csv` module, not `split(",")` · `DictReader`/`DictWriter` with
`writeheader()` · `newline=""` on every csv `open` · everything from CSV is a string, so convert at
the boundary · JSON keeps types but turns tuples into lists and keys into strings · report the row
number when you skip a row.

## 8 · Your first class

Module 02 kept a study log in parallel lists: `names`, `dates`, `minutes`. That works exactly as long
as all three stay in step - and nothing enforces that they do.

In [ ]:
names = ["maya", "ravi", "ada"]
minutes = [135, 60, 165]

print("before:", list(zip(names, minutes)))

names.sort()                     # one list moved; the other did not
print("after :", list(zip(names, minutes)))
print("ada is now credited with", minutes[names.index("ada")], "minutes -- she studied 165")

Sorting one list silently corrupted the data. No exception, no warning: `ada` is now paired with
`135`. The records were only ever related **by position**, and a single operation broke that
relationship.

The first fix is a dict per record: `{"name": "ada", "minutes": 165}`. Now the fields travel together
and sorting is safe. But nothing stops a typo like `record["minuets"]`, nothing validates the values,
and any behaviour - "how many hours is that?" - lives in a separate function that has to be told which
dict shape to expect.

A **class** is the next step: a blueprint that bundles the data *and* the behaviour, and gives the
result a name.

In [ ]:
class Session:
    """One study session: who studied, and for how long."""

    GOAL_MINUTES = 30                      # a CLASS attribute: one value, shared by all instances

    def __init__(self, name, minutes):
        """Store the two facts a session is made of."""
        self.name = name                   # INSTANCE attributes: one per object
        self.minutes = minutes

    def hours(self):
        """Return this session's length in hours."""
        return self.minutes / 60

    def met_goal(self):
        """Return True when this session reached the shared goal."""
        return self.minutes >= Session.GOAL_MINUTES

    def describe(self):
        """Return a one-line description, reusing this object's other methods."""
        verdict = "met the goal" if self.met_goal() else "under the goal"
        return f"{self.name}: {self.hours():.2f} h, {verdict}"

    def __repr__(self):
        """Return the text Python shows when it displays this object."""
        return f"Session({self.name!r}, {self.minutes})"


first = Session("maya", 45)             # INSTANTIATION: this calls __init__
second = Session("ravi", 20)

print(first)
print(second)
print(first.name, "studied", first.minutes, "minutes =", first.hours(), "hours")
print(first.describe())
print(second.describe())
print("type of first:", type(first).__name__)

Vocabulary, all of it visible above:

- the **class** `Session` is the blueprint; writing it creates no sessions;
- an **instance** is one object built from it, and `Session("maya", 45)` is an **instantiation**;
- **attributes** are the data (`first.name`, `first.minutes`);
- **methods** are the behaviour (`first.hours()`, `first.describe()`); and both are reached with a dot.

`__init__` is not a constructor you call - Python calls it for you at instantiation, with the
arguments you passed. Its job is to assign the attributes.

`self` is **the instance the method was called on**. `first.hours()` is really
`Session.hours(first)`: the dot supplies the first argument. That is why `self` appears in every
definition and never in a call, and it is why `self.` is required to reach an attribute from inside a
method.

You have been using objects since Module 01. `"text".upper()`, `scores.append(5)`, `ages.get("ada")` -
every one of those is a method on an instance.

In [ ]:
# Proof that the dot just supplies the first argument.
print("first.hours()        ->", first.hours())
print("Session.hours(first) ->", Session.hours(first))
print("same answer?", first.hours() == Session.hours(first))

# Three ways to get this wrong, all of them common.
class Broken:
    """Every method here has a defect."""

    def no_self():                          # forgot self in the DEFINITION
        return "never reachable from an instance"

    def forgot_the_dot(self):
        return name                         # meant self.name

    def __init__(self, name):
        self.name = name


broken = Broken("maya")
show_error(broken.no_self)                  # TypeError: takes 0 positional arguments but 1 was given
show_error(broken.forgot_the_dot)           # NameError: name 'name' is not defined
show_error(get_attribute, broken, "minutes")   # AttributeError: nothing ever assigned it

Three messages worth recognising on sight:

- `TypeError: Broken.no_self() takes 0 positional arguments but 1 was given` - the "1" is the instance
  the dot passed in. **You forgot `self` in the definition.**
- `NameError: name 'name' is not defined` - inside a method, a bare `name` is not the attribute. You
  need `self.name`. Python makes `self` explicit precisely so that this distinction is visible.
- `AttributeError: 'Broken' object has no attribute 'minutes'` - **attributes exist because something
  assigned to them.** Half of all `AttributeError`s are a story about order, not spelling: the
  assignment has not happened yet.

### Predict before running · one list for everybody

Here is a class with a class attribute that is a **list**:

```python
class LearnerBad:
    sessions = []              # created once, when the class was defined

    def __init__(self, name):
        self.name = name
```

Two learners are created, and `45` is appended to the first one's `sessions`.

Predict what `b.sessions` holds.

In [ ]:
# My prediction: a.sessions is ________ and b.sessions is ________

class LearnerBad:
    """A class attribute that is mutable, which is a bug."""

    GOAL = 30
    sessions = []                     # ONE list, on the class

    def __init__(self, name):
        self.name = name


a = LearnerBad("maya")
b = LearnerBad("ravi")

a.sessions.append(45)

print("a.sessions:", a.sessions)
print("b.sessions:", b.sessions)
print("the same object?", a.sessions is b.sessions)
print("id(a.sessions)        :", id(a.sessions))
print("id(b.sessions)        :", id(b.sessions))
print("id(LearnerBad.sessions):", id(LearnerBad.sessions))

`b.sessions` is `[45]`, and all three `id()` values are identical: there is exactly **one** list, it
lives on the class, and every instance is looking at the same object.

This is not a new rule. It is Module 02's aliasing, wearing a class: `b = a` never copies, and neither
does an attribute lookup that falls through to the class. See
[Module 02 · mutability](../02-python-functions-data-structures/index.html#mutability).

`GOAL = 30` as a class attribute is *fine* - an int is immutable, so nothing can be changed through
it. The bug is specific to mutable values: lists, dicts, and sets.

In [ ]:
class LearnerGood:
    """Mutable state is created per instance, inside __init__."""

    GOAL = 30                             # shared constant: fine, it is immutable

    def __init__(self, name):
        self.name = name
        self.sessions = []                # a FRESH list for every instance

    def add(self, minutes):
        """Record one session and return self, so calls can be chained."""
        self.sessions.append(minutes)
        return self

    def total(self):
        """Return the total minutes recorded."""
        return sum(self.sessions)

    def __repr__(self):
        return f"LearnerGood({self.name!r}, {self.sessions})"


c = LearnerGood("maya")
d = LearnerGood("ravi")
c.add(45).add(30).add(60)                 # chaining works because add returns self
d.add(20)

print(c, "total", c.total())
print(d, "total", d.total())
print("separate lists?", c.sessions is not d.sessions)

# The lookup rule: the instance first, then the class.
print()
print("before: c.GOAL", c.GOAL, " d.GOAL", d.GOAL, " class GOAL", LearnerGood.GOAL)
c.GOAL = 60                               # creates an INSTANCE attribute on c only
print("after : c.GOAL", c.GOAL, " d.GOAL", d.GOAL, " class GOAL", LearnerGood.GOAL)
print("c's own attributes:", sorted(vars(c)))

Two rules, both demonstrated:

1. **Create mutable state in `__init__`.** `self.sessions = []` runs once per instance, so every
   object gets its own list.
2. **Attribute lookup checks the instance first, then the class.** `c.GOAL = 60` did not change the
   class or `d`; it created a *new* attribute on `c` that shadows the class one. `vars(c)` shows only
   the instance's own attributes, which makes the difference visible.

`add` returning `self` is a small trick with a large future: it is what makes `c.add(45).add(30)`
legal, and it is exactly why scikit-learn's `fit` returns `self` (Section 9).

In [ ]:
# Dunder methods: the hooks Python calls for you.
class Minutes:
    """A number of minutes that knows how to print, compare, and add itself."""

    def __init__(self, value):
        self.value = value

    def __repr__(self):
        """For YOU: unambiguous, ideally looks like the code that would recreate it."""
        return f"Minutes({self.value})"

    def __str__(self):
        """For the USER: readable. print() prefers this one."""
        return f"{self.value} minutes"

    def __eq__(self, other):
        return isinstance(other, Minutes) and self.value == other.value

    def __add__(self, other):
        return Minutes(self.value + other.value)

    def __len__(self):
        return self.value


class NoRepr:
    """The same idea with no __repr__, for contrast."""

    def __init__(self, value):
        self.value = value


x = Minutes(45)
y = Minutes(30)

print("print uses __str__ :", x)
print("repr uses __repr__ :", repr(x))
print("in a list, __repr__:", [x, y])
print("x + y              :", x + y, "->", repr(x + y))
print("x == Minutes(45)   :", x == Minutes(45))
print("len(x)             :", len(x))
print()
print("without __repr__   :", NoRepr(45))

The pattern, once and for all: **an operator or built-in is a call to a dunder method.**

| You write | Python calls |
|---|---|
| `repr(x)`, or a list showing `x` | `x.__repr__()` |
| `print(x)`, `str(x)` | `x.__str__()`, falling back to `__repr__` |
| `a + b` | `a.__add__(b)` |
| `a == b` | `a.__eq__(b)` |
| `len(x)` | `x.__len__()` |
| `x[k]` | `x.__getitem__(k)` |
| `for item in x` | `x.__iter__()` |

`NoRepr(45)` printed as `<...NoRepr object at 0x...>`, which tells you nothing and makes every debug
session slower. **Write `__repr__` on every class you define.** It costs one line and pays for itself
the first time an object appears inside a list.

This is also how NumPy works. In Module 07, `a + b` on two arrays and `a @ b` for a matrix product are
these same hooks, implemented in C. Nothing about the syntax is special-cased for the library.

**Section 8 recap:** class = blueprint, instance = thing · `__init__` assigns, `self` **is** the
instance · attributes exist because something assigned them · a mutable class attribute is shared by
everybody · `__repr__` always.

## 9 · Inheritance and the estimator shape

**Inheritance** makes a class a specialised version of another. `class Child(Parent):` gives `Child`
everything `Parent` has, and lets it replace - **override** - any part.

In [ ]:
class Learner:
    """Someone working through the course."""

    def __init__(self, name):
        self.name = name

    def describe(self):
        """Return a one-line description."""
        return f"{self.name} is learning"

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r})"


class MLLearner(Learner):
    """A learner on a specific track."""

    def __init__(self, name, track):
        super().__init__(name)              # let the parent do the part it already knows
        self.track = track                  # then add what is new

    def describe(self):                     # OVERRIDE: same name, new behaviour
        return super().describe() + f" on the {self.track} track"


plain = Learner("ravi")
ml = MLLearner("maya", "machine learning")

print(plain.describe())
print(ml.describe())
print("ml still has .name:", ml.name)
print("repr comes from the parent:", repr(ml))
print()
print("isinstance(ml, MLLearner):", isinstance(ml, MLLearner))
print("isinstance(ml, Learner)  :", isinstance(ml, Learner), "  <-- true for subclasses too")
print("isinstance(plain, MLLearner):", isinstance(plain, MLLearner))

`MLLearner` never defines `__repr__` or `name`-handling of its own, yet has both: it inherited them.
`describe` is defined twice, and the child's version wins - that is overriding.

`super().__init__(name)` calls the parent's `__init__`. Forgetting it is the classic inheritance bug:
the parent's attributes never get assigned, and you get an `AttributeError` later, far away. The
shape to memorise is *`super().__init__(...)` first, then the new attributes*.

`isinstance(ml, Learner)` is `True`. An instance of a subclass **is** an instance of the parent - which
is the whole reason catching an ancestor exception works.

In [ ]:
# The exception hierarchy from Section 4 IS this mechanism.
print("StudyLogError's parents   :", [c.__name__ for c in StudyLogError.__bases__])
print("InvalidEntryError's parents:", [c.__name__ for c in InvalidEntryError.__bases__])
print("NotFittedError's parents  :", [c.__name__ for c in NotFittedError.__bases__])
print()

error = InvalidEntryError("row 5: minutes must be a whole number, got 'twenty'")
print("isinstance of InvalidEntryError:", isinstance(error, InvalidEntryError))
print("isinstance of StudyLogError    :", isinstance(error, StudyLogError))
print("isinstance of ValueError       :", isinstance(error, ValueError))
print("isinstance of Exception        :", isinstance(error, Exception))


def raise_invalid():
    raise InvalidEntryError("row 5: minutes must be a whole number, got 'twenty'")


check_raises("callers may catch StudyLogError", StudyLogError, raise_invalid)
check_raises("or ValueError", ValueError, raise_invalid)
check_raises("or Exception, if they must", Exception, raise_invalid)
check("NotFittedError is deliberately NOT a ValueError",
      issubclass(NotFittedError, ValueError), False)

`InvalidEntryError(StudyLogError, ValueError)` has two parents, and every `isinstance` check above is
`True`. That is not a trick - it is the point. A caller who has never heard of this project catches
`ValueError` and it works; a caller who wants everything the project raises catches `StudyLogError`
and that works too.

`NotFittedError` deliberately inherits only from `StudyLogError`. Calling `predict()` before `fit()` is
not a bad *value*; it is the wrong order of operations, so `except ValueError:` should not quietly
absorb it.

In [ ]:
# Composition: StudyLog HAS a list of entries. It is not a KIND of list.
class Note:
    """A tiny entry type, just enough to show composition."""

    def __init__(self, name, minutes):
        self.name = name
        self.minutes = minutes

    def __repr__(self):
        return f"Note({self.name!r}, {self.minutes})"


class NoteLog:
    """A collection of Notes, with behaviour of its own."""

    def __init__(self, notes=None):
        self.notes = list(notes) if notes else []      # never a mutable default argument

    def add(self, note):
        """Add one note and return self."""
        self.notes.append(note)
        return self

    def names(self):
        """Return the distinct names, in first-seen order."""
        found = []
        for note in self.notes:
            if note.name not in found:
                found.append(note.name)
        return found

    def total_minutes(self, name=None):
        """Return total minutes, for one name or for everybody."""
        chosen = self.notes if name is None else [n for n in self.notes if n.name == name]
        return sum(n.minutes for n in chosen)

    def __len__(self):
        return len(self.notes)

    def __repr__(self):
        return f"NoteLog({len(self.notes)} notes)"


log = NoteLog()
log.add(Note("maya", 45)).add(Note("maya", 90)).add(Note("ravi", 60))

print(log, "->", log.notes)
check("len() via __len__", len(log), 3)
check("names in first-seen order", log.names(), ["maya", "ravi"])
check("maya's minutes", log.total_minutes("maya"), 135)
check("everybody's minutes", log.total_minutes(), 195)

`NoteLog` **has** a list; it does not inherit from `list`. That is **composition**, and it is the right
default. Inheriting from `list` would expose `sort`, `pop`, `extend`, and forty other methods that
have nothing to do with a study log, and any of them could break your invariants. "Has-a" beats
"is-a" unless the child genuinely *is* a specialised kind of the parent.

Note `def __init__(self, notes=None)` rather than `notes=[]`: Module 02's mutable-default trap applies
to methods exactly as it does to functions.

**Duck typing** is the last piece. Python does not ask "what class is this?"; it asks "does it have the
method I need?" Any object with `.minutes` works in `total_minutes`. This is precisely why
scikit-learn will accept the class you are about to write: it looks for `fit` and `predict`, not for a
particular base class.

### The shape of every model you will ever use

Three lines, and the whole course leads to them:

```python
model = SomeModel()          # choose
model.fit(training_data)     # learn
model.predict()              # use
```

Two kinds of attribute live on that object, and telling them apart is **the single most important idea
in this module**:

- a **hyperparameter** is a choice *you* made. It is stored by `__init__`, and no data influences it.
  Here: `strategy`.
- a **fitted attribute** is something the *data* taught. It is created by `fit`, and by convention its
  name ends with a **trailing underscore**. Here: `mean_`.

scikit-learn follows this everywhere: `coef_`, `intercept_`, `n_features_in_`, `classes_`. The
underscore *is* the documentation - it tells you the value did not exist before `fit`.

In [ ]:
import statistics


class MeanMinutesModel:
    """Predict every session's minutes with one summary number.

    A real baseline: Module 11 compares a trained regression against exactly this.
    """

    def __init__(self, strategy="mean"):
        """Store the CHOICE. Nothing has been learned yet, so there is no mean_ here."""
        self.strategy = strategy

    def fit(self, entries):
        """Learn one number from entries, store it as mean_, and return self."""
        values = [entry.minutes for entry in entries]
        if not values:
            raise InvalidEntryError("cannot fit on an empty list of entries")
        if self.strategy == "mean":
            self.mean_ = sum(values) / len(values)
        elif self.strategy == "median":
            self.mean_ = statistics.median(values)
        else:
            raise StudyLogError(f"unknown strategy: {self.strategy!r}")
        return self                                    # <-- what makes chaining legal

    def predict(self):
        """Return the learned number, or raise NotFittedError if fit has not run."""
        if not hasattr(self, "mean_"):
            raise NotFittedError("call fit(entries) before predict()")
        return self.mean_

    def score(self, entries):
        """Return the mean absolute error of predict() over entries. Lower is better."""
        prediction = self.predict()
        values = [entry.minutes for entry in entries]
        if not values:
            raise InvalidEntryError("cannot score an empty list of entries")
        return sum(abs(value - prediction) for value in values) / len(values)

    def __repr__(self):
        return f"MeanMinutesModel(strategy={self.strategy!r})"


three = [Note("maya", 45), Note("maya", 30), Note("maya", 60)]

model = MeanMinutesModel()
print("fresh model      :", model)
print("its attributes   :", sorted(vars(model)), "  <-- no mean_ yet")
print("fit returns      :", model.fit(three))
print("its attributes   :", sorted(vars(model)), "  <-- mean_ appeared")
print("predict()        :", model.predict())
print("score()          :", model.score(three))

Read `vars(model)` before and after: `['strategy']`, then `['mean_', 'strategy']`. `__init__` stored
what you chose; `fit` created what the data taught. The trailing underscore on `mean_` is how a reader
knows which is which without opening the source.

`fit` printed as `MeanMinutesModel(strategy='mean')` because **it returned `self`** - the model itself,
not the mean. Every scikit-learn `fit` does this, and the reason is in the next cell.

`predict()` returned `45.0`, the mean of 45, 30, and 60. `score()` returned `10.0`: the mean absolute
error, `(|45-45| + |30-45| + |60-45|) / 3`. A baseline that always predicts the same number is a real
model - a bad one, deliberately - and every later model must beat its score to be worth anything.

In [ ]:
# 'fit returns self' is what makes this one-liner legal.
prediction = MeanMinutesModel().fit(three).predict()
check("chained fit().predict()", prediction, 45.0)

fresh = MeanMinutesModel()
returned = fresh.fit(three)
check("fit returned the SAME object, not a copy", returned is fresh, True)
print("returned is fresh?", returned is fresh, "| id match:", id(returned) == id(fresh))

# strategy changes what fit stores -- because you chose it, not because the data did.
mean_model = MeanMinutesModel().fit(three)
median_model = MeanMinutesModel(strategy="median").fit(three)
print()
print(mean_model, "-> mean_ =", mean_model.mean_)
print(median_model, "-> mean_ =", median_model.mean_)
check("mean strategy", mean_model.predict(), 45.0)
check("median strategy", median_model.predict(), 45)

# The full sample log: eight entries, mean 45.0, mean absolute error 22.5.
eight = [Note(e["name"], e["minutes"]) for e in entries]
full = MeanMinutesModel().fit(eight)
check("mean_ over all eight entries", full.mean_, 45.0)
check("score over all eight entries", full.score(eight), 22.5)

`MeanMinutesModel().fit(three).predict()` works because `fit` handed back the model. Without
`return self` that line would be `AttributeError: 'NoneType' object has no attribute 'predict'` - a
message you will now recognise instantly.

`returned is fresh` is `True`: `fit` **mutates the model in place** and returns it. It does not build a
new one. That matters when you fit twice and wonder why the old numbers are gone.

Changing `strategy` to `"median"` changed what `fit` stored - but `strategy` was still *chosen* by you
and `mean_` was still *learned* from the data. Over all eight sample entries, `mean_` is `45.0` and
`score` is `22.5`, which is the number Module 11's regression will have to beat.

In [ ]:
# Failing well: predict() before fit() must raise, not return None.
unfitted = MeanMinutesModel()

check_raises("predict() before fit()", NotFittedError, unfitted.predict)
check_raises("score() before fit()", NotFittedError, unfitted.score, three)
check_raises("and NotFittedError is a StudyLogError", StudyLogError, unfitted.predict)
print()

# The bare attribute, for contrast:
show_error(get_attribute, unfitted, "mean_")
print()

# An unknown strategy is a project-level failure, and an empty fit is an invalid entry.
check_raises("an unknown strategy", StudyLogError, MeanMinutesModel(strategy="mode").fit, three)
check_raises("fitting on nothing", InvalidEntryError, MeanMinutesModel().fit, [])

Compare the two failures carefully, because the difference is the entire argument for raising your own
exceptions.

- `unfitted.mean_` gives `AttributeError: 'MeanMinutesModel' object has no attribute 'mean_'`. True,
  generic, and it does not tell you what to do.
- `unfitted.predict()` gives `NotFittedError: call fit(entries) before predict()`. Same underlying
  situation, but the message is an instruction, and callers can catch it by name.

That is why `predict` bothers to check `hasattr` and raise. It costs two lines and converts a puzzling
symptom into an actionable one. (Making the bare `mean_` attribute itself raise would need `@property`,
which this course deliberately does not cover - so `AttributeError` is the honest answer there.)

`score` inherits the guard for free, because it calls `predict()`.

**Section 9 recap:** `class Child(Parent)` · `super().__init__(...)` first · `isinstance` is true for
subclasses · composition ("has-a") is the default · duck typing is why sklearn accepts your class ·
`__init__` stores choices, `fit` stores learning with a trailing underscore and returns `self` ·
`predict` before `fit` raises `NotFittedError`.

## 10 · Modules and the standard library

A **module** is a Python file. `import` runs that file once and binds a name to it. A **package** is a
folder of modules, which is why `from pathlib import Path` looks the way it does.

| Form | Binds | Use it |
|---|---|---|
| `import csv` | `csv` | almost always - the origin of every name stays visible |
| `import statistics as stats` | `stats` | when the real name is long |
| `from pathlib import Path` | `Path` | for one or two names you use constantly |
| `from csv import *` | everything | **never** - you no longer know where names came from |

In [ ]:
import csv
import json
import math
import statistics as stats
import sys
from collections import Counter
from datetime import date

print("math.pi        :", math.pi, "| math.sqrt(144):", math.sqrt(144))
print("stats.mean     :", stats.mean([45, 30, 60]), "| median:", stats.median([45, 30, 60]))
print("stats.stdev    :", round(stats.stdev([45, 30, 60]), 4))
print()
print("what import bound:", type(math).__name__, "named", math.__name__)
print("where it lives   :", getattr(stats, "__file__", "(built into the interpreter)"))
print("this interpreter :", sys.executable)
print("first three places import looks:", sys.path[:3])

`import math` bound **one** name, `math`, to a module object - and `type(math).__name__` confirms that
a module is just another kind of object. `math.pi` is an attribute lookup, exactly like `first.name`
in Section 8.

`module.__file__` is the file it came from, and the standard library's files are ordinary readable
Python. `sys.path` is the list of folders `import` searches, in order. `sys.executable` is the
diagnostic for "it works in the terminal but not in Jupyter": if those two commands print different
paths, they are different environments, and a package installed in one is invisible to the other.

In [ ]:
# Write your own module, then import it.
tiny = SANDBOX / "tiny_tools.py"
tiny.write_text(
    '"""A tiny module, written by the notebook."""\n'
    "\n"
    "GOAL_MINUTES = 30\n"
    "\n"
    "\n"
    "def hours(minutes):\n"
    '    """Return minutes as hours."""\n'
    "    return minutes / 60\n"
    "\n"
    "\n"
    'print("tiny_tools: top-level code runs at import time, and __name__ here is", __name__)\n'
    "\n"
    'if __name__ == "__main__":\n'
    '    print("tiny_tools: this line runs ONLY when the file is run directly")\n',
    encoding="utf-8",
)

sys.path.append(str(SANDBOX.resolve()))          # so import can find it
import tiny_tools

print()
print("GOAL_MINUTES:", tiny_tools.GOAL_MINUTES)
print("hours(90)   :", tiny_tools.hours(90))
print("its __name__:", repr(tiny_tools.__name__))

# In a notebook or a script run directly, the top-level __name__ is "__main__".
# print(__name__)      # uncomment in Jupyter to see: __main__

Three facts, all visible in the output:

1. **The file ran.** Its `print` at top level executed during the `import`, once. Importing is not a
   declaration; it is execution.
2. **Inside the module, `__name__` is `'tiny_tools'`** - the module's own name.
3. **The `if __name__ == "__main__":` line did not print.** That is the whole trick: `__name__` is
   `"__main__"` only in the file you *ran*, so the guarded block runs when you execute the file and is
   skipped when you import it. Run `python sandbox/tiny_tools.py` in a terminal to see the guarded
   line appear.

This is the promise Module 02 made, and it is why `study_tracker.py` in this folder can be both a
program and an importable library.

One notebook-specific trap: `import` caches. Editing `tiny_tools.py` and re-running `import
tiny_tools` does **nothing** - the module is already loaded. Restart the kernel. And never name your
own file `csv.py` or `random.py`: it will shadow the standard library and the resulting errors are
baffling. `__pycache__` folders are compiled caches; ignore them, they are already gitignored.

In [ ]:
# Reach for the library before writing the loop.
words = "lists dicts lists sets lists dicts".split()

by_hand = {}
for word in words:
    by_hand[word] = by_hand.get(word, 0) + 1

with_counter = Counter(words)

print("by hand    :", by_hand)
print("Counter    :", dict(with_counter))
print("most_common:", with_counter.most_common(2))
check("same answer", dict(with_counter), by_hand)

# random, with a seed, so results are reproducible.
import random

random.seed(0)
first_run = [random.randint(1, 60) for _ in range(5)]
random.seed(0)
second_run = [random.randint(1, 60) for _ in range(5)]
print()
print("seeded runs match:", first_run == second_run, first_run)

# dates are objects, not strings.
day = date.fromisoformat("2026-03-02")
print("date       :", day, "| weekday:", day.strftime("%A"), "| iso:", day.isoformat())
print("days later :", (date.fromisoformat("2026-03-05") - day).days)

In [ ]:
# Reading unfamiliar code: type, dir, help, __doc__.
session = Session("maya", 45)

print("type      :", type(session))
print("public API:", [name for name in dir(session) if not name.startswith("_")])
print("dunders   :", [name for name in dir(session) if name.startswith("__")][:6], "...")
print("docstring :", Session.__doc__)
print("one method:", Session.hours.__doc__)
print()
help(Session.met_goal)

# And the error you will meet the moment you try to install something.
print()
show_error(__import__, "numpy")

`Counter(words)` replaced Module 02's hand-rolled counting dictionary with one call, and
`.most_common(2)` replaced the sorting that would have followed. The eight modules worth knowing now:
`math`, `statistics`, `random`, `csv`, `json`, `pathlib`, `datetime`, `collections`.
`random.seed(0)` makes a "random" sequence repeatable, which is not a contradiction but a requirement:
every experiment in this course that involves randomness sets a seed so a result can be reproduced.
And `date.fromisoformat("2026-03-02")` turns the log's date column into a real date object that knows
about weekdays and subtraction - strings cannot do arithmetic.

Four more built-ins make any object explorable without documentation:

- `type(x)` - what it is;
- `dir(x)` - every name on it; the filtered comprehension
  `[n for n in dir(x) if not n.startswith("_")]` is a Module 02 comprehension doing genuinely useful
  work, and is the fastest way to find "the method that must exist";
- `help(x)` - the signature and docstring, which is why you write docstrings;
- `x.__doc__` - the same string, as a value you can print.

The last line either printed `no exception` (NumPy is already installed here) or
`ModuleNotFoundError: No module named 'numpy'`. Either is fine: Module 07 installs it. A **virtual
environment** is a private folder of packages for one project, and `pip install numpy` puts a package
into whichever environment is active - which is why `sys.executable` is the first thing to check when
an import fails in Jupyter but works in the terminal. See
[Module 01's setup section](../01-python-fundamentals/index.html) for the installation itself.

**Section 10 recap:** a module is a file, and importing runs it once · prefer `import m` and
`from m import Name`, never `import *` · `__name__` is `"__main__"` only in the file you ran ·
`sys.path` and `sys.executable` explain most import failures · reach for `Counter`, `statistics`, and
`datetime` before writing a loop · `type`, `dir`, `help` beat guessing.

## 11 · Guided exercises with gentle checks

Fourteen exercises. Replace each `...` or `None` with your answer and run the cell: every `check`
reports `--  not attempted yet` until you do, so the notebook stays safe to run from top to bottom at
any moment.

Some exercises use `check_raises`, because "it failed correctly" is a thing worth proving.

Do not open Section 14's solutions until you have wrestled with each one. Aim for at least ten before
you look at anything.

### Exercise 1 · Read a traceback

Here is a program with its lines numbered, and the traceback it produced.

```
 1  def average(values):
 2      """Return the mean of values."""
 3      total = sum(values)
 4      return total / len(values)
 5
 6
 7  def report_for(minutes_by_name, name):
 8      chosen = minutes_by_name.get(name, [])
 9      return average(chosen)
10
11
12  log = {"maya": [45, 30, 60]}
13  print(report_for(log, "ravi"))
```

```
Traceback (most recent call last):
  File "report.py", line 13, in <module>
    print(report_for(log, "ravi"))
  File "report.py", line 9, in report_for
    return average(chosen)
  File "report.py", line 4, in average
    return total / len(values)
ZeroDivisionError: division by zero
```

Set `exercise_1` to a three-item tuple: **(the exception name, the line number where it crashed, the
letter of the root cause)**.

Root-cause options:

- **a** - line 4: dividing by a length is unsafe and should never be written.
- **b** - line 8: `.get(name, [])` turned a missing learner into an empty list and passed it on.
- **c** - line 12: the log should have contained `"ravi"`.

In [ ]:
exercise_1 = None      # replace with a tuple, e.g. ("SomeError", 99, "a")

check("Exercise 1", exercise_1, ("ZeroDivisionError", 4, "b"))

### Exercise 2 · `safe_float`

Define `safe_float(text)` returning `float(text)`, or `None` when `text` is not a number. Catch
**only** `ValueError` - nothing wider. A `TypeError` from being handed something that is not text is a
bug, and should still crash.

In [ ]:
def safe_float(text):
    """Return float(text), or None when text cannot be parsed as a number."""
    ...        # replace this line


check("Exercise 2a", safe_float("3.7"), 3.7)
check("Exercise 2b", safe_float("-2"), -2.0)
check("Exercise 2c · junk gives None", safe_float("oops"), None)
check_raises("Exercise 2d · a TypeError still escapes", TypeError, safe_float, None)

### Exercise 3 · `parse_minutes`

Define `parse_minutes(text)` returning an `int`, and raising `InvalidEntryError` for anything that is
not a whole non-negative number. Chain it: `raise InvalidEntryError(...) from error`.

**Why this function has to exist:** `int("twenty")` raises a plain `ValueError`, and a loader written
around `except InvalidEntryError:` would not catch it. Translating the low-level failure into your own
family is what lets one `except` clause cover every rejectable row.

Use the message `f"minutes must be a whole number, got {text!r}"` for junk, and
`f"minutes cannot be negative: {text!r}"` for a negative value, so your output matches
`study_tracker.py`.

In [ ]:
def parse_minutes(text):
    """Return text as a non-negative int, or raise InvalidEntryError."""
    ...        # replace this line


check("Exercise 3a", parse_minutes("45"), 45)
check("Exercise 3b · whitespace is fine", parse_minutes(" 30 "), 30)
check_raises("Exercise 3c · junk", InvalidEntryError, parse_minutes, "twenty")
check_raises("Exercise 3d · negative", InvalidEntryError, parse_minutes, "-5")
check_raises("Exercise 3e · a caller may catch ValueError", ValueError, parse_minutes, "twenty")
check_raises("Exercise 3f · or StudyLogError", StudyLogError, parse_minutes, "twenty")

### Exercise 4 · Add `else` and `finally`

The function below records the labels of the clauses that ran, in order. Add an `else` clause that
appends `"else"` and a `finally` clause that appends `"finally"`, so that both target orders are
produced.

Target: `load("30")` gives `["try", "else", "finally"]`, and `load("oops")` gives
`["except", "finally"]`.

In [ ]:
order = []


def load(text):
    """Append one label per clause that runs, in order, and return the list."""
    order.clear()
    try:
        value = int(text)
        order.append("try")
    except ValueError:
        order.append("except")
    # add an else clause here
    # add a finally clause here
    return order


check("Exercise 4a", load("30"), ["try", "else", "finally"])
check("Exercise 4b", load("oops"), ["except", "finally"])

### Exercise 5 · Chain an exception

Define `to_minutes(text)` that converts `text` with `int()`, catches `ValueError`, and raises
`InvalidEntryError` **from** the original error. The second check confirms `__cause__` was set, which
only `from error` does.

In [ ]:
def to_minutes(text):
    """Return int(text); on failure raise InvalidEntryError FROM the original ValueError."""
    ...        # replace this line


check("Exercise 5a", to_minutes("45"), 45)
check_raises("Exercise 5b", InvalidEntryError, to_minutes, "thirty")

cause_name = None
try:
    to_minutes("thirty")
except InvalidEntryError as error:
    cause_name = type(error.__cause__).__name__
except Exception as error:
    print("Exercise 5: not raising InvalidEntryError yet -", type(error).__name__)

check("Exercise 5c · __cause__ is the original ValueError", cause_name, "ValueError")

### Exercise 6 · Choose the narrowest catch

For each snippet, name the **one** exception you should catch. Answer with the exception's name as a
string, in order, in a list of four.

1. `minutes = int(row["minutes"])` where the column may contain text.
2. `latest = scores[index]` where `index` may be past the end of the list.
3. `handle = open(path, "r", encoding="utf-8")` where the file may not exist.
4. `name = row["name"]` where the column may be absent from the header.

In [ ]:
exercise_6 = None      # replace with a list of four exception names, as strings

check("Exercise 6", exercise_6,
      ["ValueError", "IndexError", "FileNotFoundError", "KeyError"])

### Exercise 7 · Build a path that always works

Set `exercise_7` to an **absolute** `Path` pointing at `sandbox/log.csv`, built with `pathlib` rather
than string concatenation. It must be absolute, so that it keeps working no matter what the working
directory is.

In [ ]:
exercise_7 = None      # replace with an absolute Path ending in sandbox/log.csv

check("Exercise 7a · absolute", None if exercise_7 is None else exercise_7.is_absolute(), True)
check("Exercise 7b · file name", None if exercise_7 is None else exercise_7.name, "log.csv")
check("Exercise 7c · parent folder", None if exercise_7 is None else exercise_7.parent.name, "sandbox")

### Exercise 8 · Round-trip a text file

Write two functions:

- `write_lines(path, lines)` - write one line per item, each ending with a newline;
- `read_lines(path)` - return the lines with their newline characters removed.

Both must pass `encoding="utf-8"` and use `with`.

In [ ]:
path_8 = SANDBOX / "exercise_8.txt"


def write_lines(path, lines):
    """Write one line per item, each ending with a newline."""
    ...        # replace this line


def read_lines(path):
    """Return the file's lines with the newline characters removed."""
    ...        # replace this line


write_lines(path_8, ["90", "85", "77"])
check("Exercise 8a · round trip", read_lines(path_8), ["90", "85", "77"])
check("Exercise 8b · the file really exists", path_8.exists(), True)

### Exercise 9 · Fix a truncation bug

`add_line` is supposed to add one line to the **end** of a file. It uses mode `"w"`, so it destroys
everything already there. Fix the one wrong character.

In [ ]:
path_9 = SANDBOX / "exercise_9.txt"
with open(path_9, "w", encoding="utf-8") as handle:      # setup: a file with one line
    handle.write("first\n")


def add_line(path, text):
    """Add one line to the END of the file, keeping what is already there."""
    with open(path, "w", encoding="utf-8") as handle:    # BUG: fix this mode
        handle.write(text + "\n")


add_line(path_9, "second")
check("Exercise 9", path_9.read_text(encoding="utf-8"), "first\nsecond\n")

### Exercise 10 · Read a CSV with `DictReader`

`sandbox/exercise_10.csv` is written for you below, and one of its rows has a bad `minutes` value.
Define `minutes_from(path)` returning a list of `int`s, skipping any row whose `minutes` cannot be
converted. Use `csv.DictReader`, `encoding="utf-8"`, and `newline=""`.

In [ ]:
path_10 = SANDBOX / "exercise_10.csv"
with open(path_10, "w", encoding="utf-8", newline="") as handle:
    handle.write("name,date,minutes,topic\n")
    handle.write("maya,2026-03-02,45,functions\n")
    handle.write("ravi,2026-03-03,twenty,lists\n")
    handle.write("ada,2026-03-04,90,comprehensions\n")


def minutes_from(path):
    """Return the minutes column as ints, skipping rows that cannot be converted."""
    ...        # replace this line


check("Exercise 10", minutes_from(path_10), [45, 90])

### Exercise 11 · `load_scores`, the skip-don't-crash loader

Define `load_scores(path)`: read the file, return every parseable line as a `float`, print a warning
for each line it skips, and never crash. Blank lines are not errors - skip them silently. Your
`safe_float` from Exercise 2 makes this shorter.

In [ ]:
path_11 = SANDBOX / "scores_messy.txt"
with open(path_11, "w", encoding="utf-8") as handle:
    handle.write("90\n85\noops\n\n77\n")


def load_scores(path):
    """Return every parseable line as a float, warning about the rest."""
    ...        # replace this line


check("Exercise 11", load_scores(path_11), [90.0, 85.0, 77.0])

### Exercise 12 · JSON round trip

First **predict**, then verify. For each value, what comes back after `json.dump` followed by
`json.load`?

- `exercise_12a` - `(3, 1)` returns as a `"tuple"` or a `"list"`?
- `exercise_12b` - the key `1` in `{1: "one"}` returns as an `"int"` or a `"str"`?
- `exercise_12c` - does `{"lists", "dicts"}` (a set) save `"fine"` or raise a `"TypeError"`?

In [ ]:
exercise_12a = None    # "tuple" or "list"
exercise_12b = None    # "int" or "str"
exercise_12c = None    # "fine" or "TypeError"

check("Exercise 12a", exercise_12a, "list")
check("Exercise 12b", exercise_12b, "str")
check("Exercise 12c", exercise_12c, "TypeError")

# Now verify it yourself.
trip_path = SANDBOX / "exercise_12.json"
with open(trip_path, "w", encoding="utf-8") as handle:
    json.dump({"point": (3, 1), "counts": {1: "one"}}, handle, indent=2)
with open(trip_path, "r", encoding="utf-8") as handle:
    came_back = json.load(handle)

print("came back:", came_back)
print("point is a", type(came_back["point"]).__name__)
print("its keys  :", list(came_back["counts"]))
show_error(json.dumps, {"topics": {"lists", "dicts"}})

### Exercise 13 · Write `StudyEntry`

Define `StudyEntry` with:

- `__init__(self, name, date, minutes, topic)` storing all four as attributes;
- validation in `__init__` raising `InvalidEntryError` for a blank name, for negative minutes, and for
  more than `1440` minutes (a whole day);
- `hours()` returning the minutes as hours; and
- a `__repr__` method - which you must **add** to the stub - returning exactly
  `StudyEntry('maya', '2026-03-02', 45, 'functions')`. Remember `!r` inside the f-string; that is what
  produces the quotation marks.

Assume `minutes` arrives as an `int` - converting text is `parse_minutes`'s job.

In [ ]:
class StudyEntry:
    """One study session: name, date, minutes, topic."""

    def __init__(self, name, date, minutes, topic):
        """Validate, then store the four fields."""
        ...        # replace this line

    def hours(self):
        """Return this session's length in hours."""
        ...        # replace this line

    # add a __repr__ method here


entry = StudyEntry("maya", "2026-03-02", 45, "functions")

check("Exercise 13a · name", getattr(entry, "name", None), "maya")
check("Exercise 13b · minutes", getattr(entry, "minutes", None), 45)
check("Exercise 13c · topic", getattr(entry, "topic", None), "functions")
check("Exercise 13d · hours", entry.hours(), 0.75)
check("Exercise 13e · repr", repr(entry), "StudyEntry('maya', '2026-03-02', 45, 'functions')")
check_raises("Exercise 13f · blank name", InvalidEntryError, StudyEntry, "", "2026-03-02", 10, "x")
check_raises("Exercise 13g · negative", InvalidEntryError, StudyEntry, "maya", "2026-03-02", -5, "x")
check_raises("Exercise 13h · absurd", InvalidEntryError, StudyEntry, "maya", "2026-03-02", 5000, "x")

### Exercise 14 · Write `MeanMinutesModel`

From memory, and without scrolling back to Section 9 unless you are stuck. Define
`MeanMinutesModel` with:

- `__init__(self, strategy="mean")` storing the hyperparameter, and **no** `mean_`;
- `fit(entries)` computing the mean of `entry.minutes`, storing it as `self.mean_`, and returning
  `self`;
- `predict()` returning `mean_`, or raising `NotFittedError` when `fit` has not run.

Note what is *not* asked for: `mean_` itself does not raise. A bare `unfitted.mean_` gives Python's
generic `AttributeError`; only the methods raise the named exception. Section 12 rebuilds this class in
full, so overwriting Section 9's version here is fine.

In [ ]:
class MeanMinutesModel:
    """Predict every session's minutes with one summary number."""

    def __init__(self, strategy="mean"):
        """Store the CHOICE. Nothing is learned yet."""
        ...        # replace this line

    def fit(self, entries):
        """Learn the mean of entry.minutes, store it as mean_, and return self."""
        ...        # replace this line

    def predict(self):
        """Return mean_, or raise NotFittedError."""
        ...        # replace this line


sample = [StudyEntry("maya", "2026-03-02", 45, "functions"),
          StudyEntry("maya", "2026-03-03", 30, "lists"),
          StudyEntry("maya", "2026-03-04", 60, "dicts")]

model_14 = MeanMinutesModel()
check("Exercise 14a · strategy stored", getattr(model_14, "strategy", None), "mean")
check("Exercise 14b · no mean_ before fit", hasattr(model_14, "mean_"), False)
check_raises("Exercise 14c · predict() before fit()", NotFittedError, model_14.predict)
check("Exercise 14d · fit returns self", model_14.fit(sample) is model_14, True)
check("Exercise 14e · mean_ learned", getattr(model_14, "mean_", None), 45.0)
check("Exercise 14f · predict", model_14.predict(), 45.0)
chained = MeanMinutesModel().fit(sample)
check("Exercise 14g · chaining works", None if chained is None else chained.predict(), 45.0)

That is all fourteen. Two habits to carry forward from them:

- every failure you *expect* got a named exception and a message a human can act on; and
- every value that crossed a boundary - a file, a CSV column, a JSON round trip - was converted and
  checked at that boundary rather than three functions later.

Solutions are in Section 14. Compare yours even where the check passed: a passing answer can still be
harder to read than it needs to be.

## 12 · Mini-project: the study tracker

Module 01's project reported on one learner with one loop. Module 02's handled many learners with
functions and containers, but its data was hard-coded and it crashed on bad input. This one is
**persistent, robust, and object-shaped**: it loads a CSV from disk, refuses bad rows with a named
exception, saves a JSON summary, and exposes a scikit-learn-shaped baseline model.

**Requirements**

1. `StudyEntry` - one session: `name`, `date`, `minutes`, `topic`. Validates in `__init__`.
2. `StudyLog` - **has** a list of entries. `add`, `names`, `for_name`, `total_minutes`, `summary`.
3. Loading skips bad rows, reports each with its row number, and never crashes on a malformed file.
4. `MeanMinutesModel` - `fit` returns `self`, sets `mean_`, and `predict`/`score` raise
   `NotFittedError` when unfitted.
5. The three exceptions: `StudyLogError`, `InvalidEntryError`, `NotFittedError`.
6. `__repr__` on `StudyEntry` and `MeanMinutesModel`.
7. Every file access uses `pathlib`, `with`, and `encoding="utf-8"`; every csv call adds `newline=""`.
8. It runs twice in a row with identical results, from any working directory.

**Six milestones**

| # | Milestone | Uses |
|---|---|---|
| 1 | `StudyEntry` with `__init__` and `__repr__` | Section 8 |
| 2 | Validation that raises `InvalidEntryError` | Sections 4, 9 |
| 3 | `StudyLog` with `add` and `summary` | Section 8, plus Module 02 |
| 4 | `load_log(path)` with `DictReader`, skipping bad rows | Sections 5, 6, 7 |
| 5 | `save_summary(log, path)` writing JSON with `indent=2` | Section 7 |
| 6 | `MeanMinutesModel` with `fit`/`predict`/`score` | Section 9 |

Each cell below is one or two milestones, with its own checks. Run them in order.

In [ ]:
# ---- Milestone 1: StudyEntry, with __init__ and __repr__ -----------------
GOAL_MINUTES = 30
MAX_REASONABLE_MINUTES = 1440          # a whole day; more than this is a typo


class StudyEntry:
    """One study session: who, when, how long, and on what."""

    def __init__(self, name, date, minutes, topic):
        """Store the four fields that make up a session."""
        self.name = name
        self.date = date
        self.minutes = minutes
        self.topic = topic

    def hours(self):
        """Return this session's length in hours."""
        return self.minutes / 60

    def met_goal(self, goal=GOAL_MINUTES):
        """Return True when this session reached the daily goal."""
        return self.minutes >= goal

    def __repr__(self):
        return (f"StudyEntry({self.name!r}, {self.date!r}, "
                f"{self.minutes}, {self.topic!r})")


first = StudyEntry("maya", "2026-03-02", 45, "functions")
print(first)
check("M1 · repr shows all four fields", repr(first),
      "StudyEntry('maya', '2026-03-02', 45, 'functions')")
check("M1 · hours", first.hours(), 0.75)
check("M1 · met_goal", first.met_goal(), True)
check("M1 · met_goal with a custom goal", first.met_goal(goal=60), False)

In [ ]:
# ---- Milestone 2: validation, and the conversion that feeds it -----------
def parse_minutes(text):
    """Return text as a non-negative whole number of minutes, or raise InvalidEntryError."""
    try:
        minutes = int(str(text).strip())
    except ValueError as error:
        raise InvalidEntryError(
            f"minutes must be a whole number, got {text!r}") from error
    if minutes < 0:
        raise InvalidEntryError(f"minutes cannot be negative: {text!r}")
    return minutes


class StudyEntry(StudyEntry):          # same class, now with validation in __init__
    """One study session, validated on creation."""

    def __init__(self, name, date, minutes, topic):
        """Validate every field, then store it."""
        if not str(name).strip():
            raise InvalidEntryError("name must not be blank")
        if not isinstance(minutes, int):
            raise InvalidEntryError(f"minutes must be an int, got {type(minutes).__name__}")
        if minutes < 0:
            raise InvalidEntryError(f"minutes cannot be negative: {minutes!r}")
        if minutes > MAX_REASONABLE_MINUTES:
            raise InvalidEntryError(f"more than a day of study is not credible: {minutes!r}")
        super().__init__(str(name).strip(), date, minutes, str(topic).strip())


check("M2 · a good row still works", repr(StudyEntry("maya", "2026-03-02", 45, "functions")),
      "StudyEntry('maya', '2026-03-02', 45, 'functions')")
check("M2 · parse_minutes", parse_minutes(" 30 "), 30)
check_raises("M2 · junk minutes", InvalidEntryError, parse_minutes, "twenty")
check_raises("M2 · negative minutes", InvalidEntryError, parse_minutes, "-5")
check_raises("M2 · blank name", InvalidEntryError, StudyEntry, "", "2026-03-02", 10, "x")
check_raises("M2 · negative", InvalidEntryError, StudyEntry, "maya", "2026-03-02", -5, "x")
check_raises("M2 · absurd", InvalidEntryError, StudyEntry, "maya", "2026-03-02", 5000, "x")
check_raises("M2 · callers may catch ValueError", ValueError, StudyEntry, "", "2026-03-02", 10, "x")

In [ ]:
# ---- Milestone 3: StudyLog -- composition, not inheritance ---------------
class StudyLog:
    """A collection of StudyEntry objects, with the questions we ask of it."""

    def __init__(self, entries=None):
        """Start from an existing iterable of entries, or empty."""
        self.entries = list(entries) if entries else []

    def add(self, entry):
        """Add one entry and return self, so calls can be chained."""
        self.entries.append(entry)
        return self

    def names(self):
        """Return the distinct learner names, in first-seen order."""
        found = []
        for entry in self.entries:
            if entry.name not in found:
                found.append(entry.name)
        return found

    def for_name(self, name):
        """Return every entry belonging to one learner."""
        return [entry for entry in self.entries if entry.name == name]

    def total_minutes(self, name=None):
        """Return total minutes for one learner, or for everybody."""
        chosen = self.entries if name is None else self.for_name(name)
        return sum(entry.minutes for entry in chosen)

    def summary(self):
        """Return {name: {sessions, total, average, best, days_at_goal}}."""
        result = {}
        for name in self.names():
            mine = self.for_name(name)
            minutes = [entry.minutes for entry in mine]
            result[name] = {
                "sessions": len(mine),
                "total": sum(minutes),
                "average": sum(minutes) / len(minutes),
                "best": max(minutes),
                "days_at_goal": len([m for m in minutes if m >= GOAL_MINUTES]),
            }
        return result

    def __len__(self):
        return len(self.entries)

    def __repr__(self):
        return f"StudyLog({len(self.entries)} entries)"


def class_totals(log):
    """Return (learners, minutes, sessions_at_goal) for the whole log."""
    at_goal = len([entry for entry in log.entries if entry.met_goal()])
    return len(log.names()), log.total_minutes(), at_goal


demo = StudyLog().add(StudyEntry("maya", "2026-03-02", 45, "functions"))
demo.add(StudyEntry("ravi", "2026-03-02", 20, "functions"))
print(demo, "->", demo.names())
check("M3 · len() via __len__", len(demo), 2)
check("M3 · total for one name", demo.total_minutes("maya"), 45)
check("M3 · total for everybody", demo.total_minutes(), 65)
check("M3 · summary keys", sorted(demo.summary()["maya"]),
      ["average", "best", "days_at_goal", "sessions", "total"])

In [ ]:
# ---- Milestone 4: load_log -- DictReader, skipping bad rows --------------
def load_log(path):
    """Return a StudyLog read from a CSV, skipping and reporting unusable rows."""
    path = Path(path)
    try:
        handle = open(path, "r", encoding="utf-8", newline="")
    except FileNotFoundError as error:
        raise FileNotFoundError(f"no study log at {path.resolve()}") from error

    log = StudyLog()
    with handle:
        reader = csv.DictReader(handle)
        for row_number, row in enumerate(reader, start=2):     # line 1 is the header
            try:
                minutes = parse_minutes(row["minutes"])
                log.add(StudyEntry(row["name"], row["date"], minutes, row["topic"]))
            except InvalidEntryError as error:
                print(f"skipping row {row_number}: {error}")
    return log


log = load_log(SAMPLE_LOG)
print(log)
check("M4 · eight entries", len(log), 8)
check("M4 · names in first-seen order", log.names(), ["maya", "ravi", "ada"])
check("M4 · maya's total", log.total_minutes("maya"), 135)
check("M4 · ravi's total", log.total_minutes("ravi"), 60)
check("M4 · ada's total", log.total_minutes("ada"), 165)
check("M4 · ravi's average", log.summary()["ravi"]["average"], 20.0)
check("M4 · maya's average", log.summary()["maya"]["average"], 45.0)
check("M4 · ada's average", log.summary()["ada"]["average"], 82.5)
check("M4 · maya's best day", log.summary()["maya"]["best"], 60)
check("M4 · maya's days at goal", log.summary()["maya"]["days_at_goal"], 3)
check("M4 · ravi's days at goal", log.summary()["ravi"]["days_at_goal"], 0)
check("M4 · class totals", class_totals(log), (3, 360, 5))

print()
partial = load_log(one_bad)                     # the file with 'twenty' in row 5
check("M4 · one bad row costs one row, not the run", len(partial), 7)
print()
check_raises("M4 · a missing file", FileNotFoundError, load_log, SANDBOX / "nope.csv")

In [ ]:
# ---- Milestone 5: save_summary -- JSON with indent=2 ---------------------
def save_summary(log, path):
    """Write log.summary() to path as JSON, and return the path."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(log.summary(), handle, indent=2)
    return path


summary_path = save_summary(log, SANDBOX / "summary.json")
print(summary_path.read_text(encoding="utf-8")[:220], "...")

with open(summary_path, "r", encoding="utf-8") as handle:
    reloaded = json.load(handle)

check("M5 · the round trip is equal", reloaded, log.summary())
check("M5 · ravi's average survived", reloaded["ravi"]["average"], 20.0)
check("M5 · the keys are learner names", sorted(reloaded), ["ada", "maya", "ravi"])

In [ ]:
# ---- Milestone 6: MeanMinutesModel --------------------------------------
class MeanMinutesModel:
    """Predict every session's minutes with one summary number.

    __init__ stores the hyperparameter `strategy`. fit creates the fitted
    attribute `mean_` and returns self. Module 11 compares a real regression
    against exactly this baseline.
    """

    def __init__(self, strategy="mean"):
        """Store the choice. Nothing has been learned yet."""
        self.strategy = strategy

    def fit(self, entries):
        """Learn one number from entries, store it as mean_, and return self."""
        values = [entry.minutes for entry in entries]
        if not values:
            raise InvalidEntryError("cannot fit on an empty list of entries")
        if self.strategy == "mean":
            self.mean_ = sum(values) / len(values)
        elif self.strategy == "median":
            self.mean_ = statistics.median(values)
        else:
            raise StudyLogError(f"unknown strategy: {self.strategy!r}")
        return self

    def predict(self):
        """Return the learned number, or raise NotFittedError."""
        if not hasattr(self, "mean_"):
            raise NotFittedError("call fit(entries) before predict()")
        return self.mean_

    def score(self, entries):
        """Return the mean absolute error of predict() over entries. Lower is better."""
        prediction = self.predict()
        values = [entry.minutes for entry in entries]
        if not values:
            raise InvalidEntryError("cannot score an empty list of entries")
        return sum(abs(value - prediction) for value in values) / len(values)

    def __repr__(self):
        return f"MeanMinutesModel(strategy={self.strategy!r})"


check_raises("M6 · predict() before fit()", NotFittedError, MeanMinutesModel().predict)
check_raises("M6 · score() before fit()", NotFittedError, MeanMinutesModel().score, log.entries)

model = MeanMinutesModel()
check("M6 · fit returns the model itself", model.fit(log.entries) is model, True)
check("M6 · mean_ over the whole log", model.mean_, 45.0)
check("M6 · predict", model.predict(), 45.0)
check("M6 · score (mean absolute error)", model.score(log.entries), 22.5)

mayas = log.for_name("maya")
check("M6 · predict for maya's three sessions", MeanMinutesModel().fit(mayas).predict(), 45.0)
check("M6 · score for maya's three sessions", MeanMinutesModel().fit(mayas).score(mayas), 10.0)
check("M6 · the median strategy is supported", MeanMinutesModel(strategy="median").fit(mayas).mean_, 45)
check_raises("M6 · an unknown strategy", StudyLogError, MeanMinutesModel(strategy="mode").fit, mayas)
print(model)

In [ ]:
# ---- The report, and the twice-in-a-row requirement ---------------------
def print_report(log):
    """Display the whole report. The only function here with a side effect."""
    summary = log.summary()
    print("STUDY LOG REPORT")
    print("=" * 52)
    ranked = sorted(summary, key=lambda name: summary[name]["total"], reverse=True)
    for name in ranked:
        s = summary[name]
        print(f"{name:<6}{s['sessions']:>3} sessions {s['total']:>4} min  "
              f"avg {s['average']:>5.1f}  best {s['best']:>3}  at goal {s['days_at_goal']}")
    print("-" * 52)
    learners, minutes, at_goal = class_totals(log)
    print(f"{learners} learners, {minutes} minutes, {at_goal} sessions at or above "
          f"{GOAL_MINUTES} minutes")
    fitted = MeanMinutesModel().fit(log.entries)
    print(f"baseline {fitted!r}: predicts {fitted.predict():.1f} min, "
          f"mean absolute error {fitted.score(log.entries):.1f}")


def run_once():
    """Load, report, and save. Returns the summary, for comparison."""
    fresh = load_log(SAMPLE_LOG)
    print_report(fresh)
    save_summary(fresh, SANDBOX / "summary.json")
    return fresh.summary()


first_run = run_once()
print()
second_run = run_once()
print()
check("running twice gives identical results", first_run, second_run)

### Test checklist

Tick each one off against the output above, or write the call yourself.

| | Test | Expected |
|---|---|---|
| [ ] | `StudyEntry("maya", "2026-03-02", 45, "functions")` | repr shows all four fields |
| [ ] | `StudyEntry("maya", "2026-03-02", -5, "x")` | raises `InvalidEntryError` |
| [ ] | `StudyEntry("", "2026-03-02", 10, "x")` | raises `InvalidEntryError` |
| [ ] | `load_log(SAMPLE_LOG)` | 8 entries, no crash |
| [ ] | a CSV with one bad row | 7 entries plus one reported skip, still returns |
| [ ] | `load_log("nope.csv")` | `FileNotFoundError` naming the resolved path |
| [ ] | `log.total_minutes("maya")` | 135 |
| [ ] | `log.summary()["ravi"]["average"]` | 20.0 |
| [ ] | `save_summary(log, p)` then reading `p` back | an equal dict |
| [ ] | `MeanMinutesModel().predict()` | `NotFittedError` |
| [ ] | `MeanMinutesModel().fit(entries)` | returns the model itself (`is` the same object) |
| [ ] | `fit` then `predict()` on `[45, 30, 60]` | 45.0 |
| [ ] | running it twice | identical output |

**Rubric, out of 10.** Correctness 3 · Error design 2 (named exceptions, narrow catches, no silent
`pass`) · File handling 2 (`pathlib`, `with`, `encoding`, `newline`) · Object design 1 (composition,
`__repr__`, no shared mutable class attribute) · Estimator shape 1 (`fit` returns `self`, `mean_` has
its underscore, `NotFittedError`) · Explanation 1 (you can trace one `load_log` call and name the root
cause of one skipped row). Aim for 8, including all three correctness points.

### From notebook to script

Open [`study_tracker.py`](study_tracker.py) in this folder and run it from a terminal:

```
cd modules\03-python-oop-errors-files
python study_tracker.py
```

It prints the same report, demonstrates one deliberately rejected row, saves a JSON summary, and fits
the baseline model. Four things in it are worth studying:

- it builds every path as `Path(__file__).parent / "sample_log.csv"`, so it works from **any** working
  directory - try running it from the repository root and see;
- it writes only into `sandbox/`, created with `mkdir(exist_ok=True)`;
- it is importable with **no side effects**, thanks to `if __name__ == "__main__": main()`. Try
  `from study_tracker import StudyLog` in a cell: nothing prints; and
- the definitions come first, the side-effecting functions (`load_log`, `save_summary`,
  `print_report`, `main`) come last, and exactly one call starts everything.

That last point is the shape of every Python program you will write from here on.

## 13 · Optional challenges

Attempt these after the exercises and the project. Each one combines several ideas, and none has a
`check` waiting - decide for yourself when it is right, which is the skill being practised.

### Challenge A · `Vector2` - the bridge to Module 04

Write a `Vector2` class with:

- `__init__(self, x, y)`;
- `dot(self, other)` returning `self.x * other.x + self.y * other.y`;
- `__repr__` returning `Vector2(3, 1)`;
- `__add__` returning a new `Vector2`; and
- `__eq__` so that `Vector2(3, 1) == Vector2(3, 1)` is `True`.

```python
u = Vector2(3, 1)
v = Vector2(1, 2)
print(u.dot(v))          # 5
print(u + v)             # Vector2(4, 3)
print(u == Vector2(3, 1))  # True
```

The dot product you are about to implement is the star of the next module, and in Module 07 NumPy will
give you the same operation on a million numbers at once - through exactly these dunder hooks.

In [ ]:
# Challenge A: your Vector2 here.

### Challenge B · `Counter` versus the hand-rolled loop

Take Module 02's word count and write it twice: once with a plain dict and `.get()`, once with
`collections.Counter`. Then answer three questions in a comment:

1. Which version would you rather read in six months?
2. `Counter` is a subclass of `dict` - check with `issubclass`. What does that buy it?
3. Name one thing the hand-rolled version makes easy that `Counter` does not.

```python
text = "lists dicts lists sets lists dicts"
```

In [ ]:
# Challenge B: both versions, plus your three answers as comments.

### Challenge C · A retry wrapper

Write `retry(func, attempts=3)` that calls `func()` and, if it raises, tries again - up to `attempts`
times in total. If the last attempt also fails, raise `StudyLogError("all attempts failed")`
**`from`** the final exception, so the traceback shows the real cause.

Test it with a function that fails the first two times and succeeds on the third, using a counter in
an enclosing list. Then test it with one that always fails, and read the chained traceback.

Two design questions to answer in comments: which exception types should a retry wrapper catch (and
which must it never catch), and why is retrying a `ValueError` from `int("thirty")` pointless?

In [ ]:
# Challenge C: your retry wrapper here.

### Challenge D · Merge two logs

1. Write a second CSV into `sandbox/` with a few rows, at least one of which duplicates a
   `(name, date)` pair from `sample_log.csv`.
2. Load both with `load_log`.
3. Merge them into one `StudyLog`, **deduplicating on `(name, date)`** - keep the first occurrence.
   A `set` of tuples is the tool; this is why tuples are hashable and lists are not.
4. Save the merged summary as JSON with `indent=2`, and check that the totals changed by exactly what
   you expect.

Extension: report the duplicates you dropped, with their row numbers, in the same style as
`load_log`'s skip messages.

In [ ]:
# Challenge D: your merge here.

<details>
<summary><strong>Open solutions only after attempting the exercises</strong></summary>

## 14 · Solutions

### Exercise 1

```python
exercise_1 = ("ZeroDivisionError", 4, "b")
```

The last line names the exception; the frame directly above it names line 4 in `average`, which is the
**crash site**. The root cause is line 8: `.get(name, [])` converted "no such learner" into an empty
list and passed it on as if it were data. Option **a** is wrong because dividing by a length is normal
code; option **c** confuses "the data was not what I wanted" with "the code mishandled the data".

### Exercise 2

```python
def safe_float(text):
    """Return float(text), or None when text cannot be parsed as a number."""
    try:
        return float(text)
    except ValueError:
        return None
```

`float(None)` raises `TypeError`, which this deliberately does not catch: being handed `None` where
text was expected is a bug in the caller, not an expected failure.

### Exercise 3

```python
def parse_minutes(text):
    """Return text as a non-negative int, or raise InvalidEntryError."""
    try:
        minutes = int(str(text).strip())
    except ValueError as error:
        raise InvalidEntryError(f"minutes must be a whole number, got {text!r}") from error
    if minutes < 0:
        raise InvalidEntryError(f"minutes cannot be negative: {text!r}")
    return minutes
```

The `from error` matters: without it the traceback says "During handling of the above exception,
another exception occurred", which reads like an accident rather than a decision. And because
`InvalidEntryError` inherits from both `StudyLogError` and `ValueError`, all three of checks 3c, 3e and
3f pass on the same object.

### Exercise 4

```python
def load(text):
    order.clear()
    try:
        value = int(text)
        order.append("try")
    except ValueError:
        order.append("except")
    else:
        order.append("else")
    finally:
        order.append("finally")
    return order
```

`else` runs only when `try` finished without an exception; `finally` runs in both cases, and always
last.

### Exercise 5

```python
def to_minutes(text):
    """Return int(text); on failure raise InvalidEntryError FROM the original ValueError."""
    try:
        return int(text)
    except ValueError as error:
        raise InvalidEntryError(f"bad minutes: {text!r}") from error
```

`error.__cause__` is the original `ValueError`. Drop the `from error` and `__cause__` is `None` while
`__context__` still holds it - which is the difference between "I replaced this deliberately" and
"something went wrong while I was handling something else".

### Exercise 6

```python
exercise_6 = ["ValueError", "IndexError", "FileNotFoundError", "KeyError"]
```

Every one is the narrowest name available. `int()` on text that is not a number is a `ValueError`, not
a `TypeError` - a string is the right kind of thing. `except OSError:` would also catch case 3, but it
also catches permission problems and a dozen unrelated failures, so it is the wrong width unless you
truly mean "anything wrong with the file".

### Exercise 7

```python
exercise_7 = (SANDBOX / "log.csv").resolve()
```

`.resolve()` is what makes it absolute. In a script you would write
`Path(__file__).parent / "sandbox" / "log.csv"`, which needs no `resolve()` because `__file__` is
already absolute in every modern Python.

### Exercise 8

```python
def write_lines(path, lines):
    """Write one line per item, each ending with a newline."""
    with open(path, "w", encoding="utf-8") as handle:
        for line in lines:
            handle.write(line + "\n")


def read_lines(path):
    """Return the file's lines with the newline characters removed."""
    with open(path, "r", encoding="utf-8") as handle:
        return [line.strip() for line in handle]
```

`handle.writelines(line + "\n" for line in lines)` is an equally good one-liner. What does *not* work
is `handle.writelines(lines)`: it adds no separators at all.

### Exercise 9

```python
with open(path, "a", encoding="utf-8") as handle:      # "a", not "w"
```

One character. `"w"` empties the file the instant it opens, before your `write` ever runs.

### Exercise 10

```python
def minutes_from(path):
    """Return the minutes column as ints, skipping rows that cannot be converted."""
    values = []
    with open(path, "r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        for row_number, row in enumerate(reader, start=2):
            try:
                values.append(int(row["minutes"]))
            except ValueError:
                print(f"skipping row {row_number}: minutes is {row['minutes']!r}")
    return values
```

Only the conversion is inside the `try`. Putting the `append` in there as well would be harmless here,
but the habit of keeping the risky line alone is what stops an unrelated failure from being silently
treated as a bad row.

### Exercise 11

```python
def load_scores(path):
    """Return every parseable line as a float, warning about the rest."""
    scores = []
    with open(path, "r", encoding="utf-8") as handle:
        for number, line in enumerate(handle, start=1):
            text = line.strip()
            if not text:
                continue
            value = safe_float(text)
            if value is None:
                print(f"skipping line {number}: {text!r} is not a number")
            else:
                scores.append(value)
    return scores
```

`safe_float` from Exercise 2 does the risky part, so this function contains no `try` at all - small
functions composing into a larger one, which is Module 02's whole argument.

### Exercise 12

```python
exercise_12a = "list"        # JSON has one sequence type
exercise_12b = "str"         # JSON object keys are always text
exercise_12c = "TypeError"   # Object of type set is not JSON serializable
```

The first two are silent; the third is loud. Convert a set with `sorted(my_set)` before saving.

### Exercise 13

```python
class StudyEntry:
    """One study session: name, date, minutes, topic."""

    def __init__(self, name, date, minutes, topic):
        """Validate, then store the four fields."""
        if not str(name).strip():
            raise InvalidEntryError("name must not be blank")
        if minutes < 0:
            raise InvalidEntryError(f"minutes cannot be negative: {minutes!r}")
        if minutes > 1440:
            raise InvalidEntryError(f"more than a day of study is not credible: {minutes!r}")
        self.name = name
        self.date = date
        self.minutes = minutes
        self.topic = topic

    def hours(self):
        """Return this session's length in hours."""
        return self.minutes / 60

    def __repr__(self):
        return f"StudyEntry({self.name!r}, {self.date!r}, {self.minutes}, {self.topic!r})"
```

Validate **before** assigning: a half-built object that exists but is invalid is worse than no object.
The `!r` in the f-string is what produces the quotes in the repr, and it is why the expected string is
`StudyEntry('maya', ...)` rather than `StudyEntry(maya, ...)`.

### Exercise 14

```python
class MeanMinutesModel:
    """Predict every session's minutes with one summary number."""

    def __init__(self, strategy="mean"):
        """Store the CHOICE. Nothing is learned yet."""
        self.strategy = strategy

    def fit(self, entries):
        """Learn the mean of entry.minutes, store it as mean_, and return self."""
        values = [entry.minutes for entry in entries]
        if not values:
            raise InvalidEntryError("cannot fit on an empty list of entries")
        self.mean_ = sum(values) / len(values)
        return self

    def predict(self):
        """Return mean_, or raise NotFittedError."""
        if not hasattr(self, "mean_"):
            raise NotFittedError("call fit(entries) before predict()")
        return self.mean_
```

Three details carry the whole idea: `mean_` is **not** created in `__init__`, so `hasattr` is a
truthful test; `fit` ends with `return self`, which is what makes `MeanMinutesModel().fit(x).predict()`
legal; and `predict` raises a **named** exception with an actionable message instead of letting Python
produce a generic `AttributeError`.

### Challenge A · `Vector2`

```python
class Vector2:
    """A two-dimensional vector."""

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def dot(self, other):
        """Return the dot product of this vector and another."""
        return self.x * other.x + self.y * other.y

    def __add__(self, other):
        return Vector2(self.x + other.x, self.y + other.y)

    def __eq__(self, other):
        return isinstance(other, Vector2) and (self.x, self.y) == (other.x, other.y)

    def __repr__(self):
        return f"Vector2({self.x}, {self.y})"
```

`__add__` returns a **new** `Vector2` rather than modifying `self` - the same choice as `sorted()`
versus `.sort()`. Defining `__eq__` makes instances unhashable unless you also define `__hash__`, which
is fine here because nothing uses a vector as a dict key.

### Challenge B · `Counter`

```python
text = "lists dicts lists sets lists dicts"
words = text.split()

by_hand = {}
for word in words:
    by_hand[word] = by_hand.get(word, 0) + 1

from collections import Counter
with_counter = Counter(words)

print(by_hand == dict(with_counter))          # True
print(with_counter.most_common(1))            # [('lists', 3)]
print(issubclass(Counter, dict))              # True
```

1. `Counter` - one line, no accumulator to get wrong.
2. Being a `dict` subclass means every dict method still works: `.items()`, `in`, `.get()`,
   comprehensions over it. `isinstance(with_counter, dict)` is `True`, which is Section 9's rule again.
3. The hand-rolled loop is easier to change: counting only words longer than three characters, or
   totalling minutes instead of occurrences, is a one-line edit to the loop body.

### Challenge C · A retry wrapper

```python
def retry(func, attempts=3):
    """Call func() up to `attempts` times; re-raise the last failure with context."""
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            return func()
        except (OSError, TimeoutError) as error:
            last_error = error
            print(f"attempt {attempt} failed: {type(error).__name__}: {error}")
    raise StudyLogError(f"all {attempts} attempts failed") from last_error


calls = [0]


def flaky():
    """Fail twice, then succeed."""
    calls[0] += 1
    if calls[0] < 3:
        raise OSError("temporary glitch")
    return "ok"


print(retry(flaky))          # two failure lines, then ok
```

Catch only failures that a *later* attempt could plausibly survive: file locks, network timeouts,
`OSError`. Never retry a `ValueError` from `int("thirty")` - the input is identical every time, so the
result will be too. Retrying a bug just makes it slower.

### Challenge D · Merge two logs

```python
extra = SANDBOX / "extra_log.csv"
with open(extra, "w", encoding="utf-8", newline="") as handle:
    handle.write("name,date,minutes,topic\n")
    handle.write("maya,2026-03-02,45,functions\n")     # a duplicate of the first row
    handle.write("ravi,2026-03-06,35,classes\n")


def merge(*logs):
    """Return one StudyLog, keeping the first entry seen for each (name, date)."""
    merged = StudyLog()
    seen = set()
    for log in logs:
        for entry in log.entries:
            key = (entry.name, entry.date)
            if key in seen:
                print(f"dropping duplicate {key}")
                continue
            seen.add(key)
            merged.add(entry)
    return merged


both = merge(load_log(SAMPLE_LOG), load_log(extra))
print(len(both), "entries")                    # 9: 8 + 2 - 1 duplicate
print(both.total_minutes())                    # 395: 360 + 35
save_summary(both, SANDBOX / "merged.json")
```

`(entry.name, entry.date)` works as a set element because a tuple of strings is **hashable**; a list of
the same two strings would raise `TypeError: unhashable type: 'list'`. That is Module 02's rule paying
off in a real design decision.

</details>

## 15 · Exit ticket: prove you are ready for Module 04

A performance check, not a recognition quiz. Complete all four parts without copying a nearby example.

**Pass rule:** all four parts must be independently correct.

- **A · Write** - `load_validated(path)`: read a study-log CSV and return a list of
  `(name, minutes)` pairs with `minutes` as an `int`. Raise `FileNotFoundError` for a missing file, with
  a message naming the resolved path. Review: [Part 5](index.html#csv).
- **B · Predict** - write down the printed order **before** running the cell, and it must match.
  Review: [Part 2](index.html#else-finally).
- **C · Repair** - one class, three defects from this module: a method missing `self`, a shared mutable
  class attribute, and an `except Exception: pass`. Find and fix all three. Review:
  [Part 6](index.html#class-attributes).
- **D · Explain** - one sentence each: why `mean_` has a trailing underscore and `strategy` does not.
  Review: [Part 7](index.html#estimator-api).

If one part is not yet correct, review the linked lesson part, change your work, and try that part
again.

In [ ]:
# ---- A · Write ----------------------------------------------------------
def load_validated(path):
    """Return [(name, minutes), ...] from a study-log CSV. Raise FileNotFoundError if absent."""
    ...        # replace this line


loaded = load_validated(SAMPLE_LOG)

check("Exit A · row count", None if not loaded else len(loaded), 8)
check("Exit A · first pair", None if not loaded else loaded[0], ("maya", 45))
check("Exit A · minutes are ints", None if not loaded else type(loaded[0][1]).__name__, "int")
check_raises("Exit A · missing file", FileNotFoundError, load_validated, SANDBOX / "nope.csv")

In [ ]:
# ---- B · Predict --------------------------------------------------------
# My prediction: probe("30") prints ______________ and returns ______

trail = []


def probe(text):
    """Record which clauses run, in order."""
    try:
        value = int(text)
        trail.append("try")
        return value
    except ValueError:
        trail.append("except")
        return -1
    else:
        trail.append("else")
    finally:
        trail.append("finally")


trail.clear()
returned = probe("30")
print("order:", trail, "| returned:", returned)

exit_b = None      # replace with the list you predicted, e.g. ["try", "else", "finally"]
check("Exit B", exit_b, ["try", "finally"])

In [ ]:
# ---- C · Repair --------------------------------------------------------
# Broken version. Three defects; do not edit it -- rewrite it below.
class BrokenLog:
    entries = []                          # defect 1

    def __init__(self, name):
        self.name = name

    def add(minutes):                     # defect 2
        self.entries.append(minutes)

    def total(self):
        try:
            return sum(self.entries)
        except Exception:                 # defect 3
            pass


# ---- your repaired version (keep the name FixedLog) ----
class FixedLog:
    """One learner's log, repaired."""

    def __init__(self, name):
        """Store the name, and a fresh list of entries."""
        ...        # replace this line

    def add(self, minutes):
        """Record one session and return self."""
        ...        # replace this line

    def total(self):
        """Return the total minutes recorded."""
        ...        # replace this line


p = FixedLog("maya")
q = FixedLog("ravi")
p.add(45)
p.add(30)

check("Exit C · total", p.total(), 75)
check("Exit C · no shared state", getattr(q, "entries", None), [])
check("Exit C · separate objects",
      None if not hasattr(p, "entries") else p.entries is not q.entries, True)
check("Exit C · add returns self", p.add(10) is p, True)

In [ ]:
# ---- D · Explain -------------------------------------------------------
exit_d_mean = None       # one sentence: why does mean_ end with an underscore?
exit_d_strategy = None   # one sentence: why does strategy not?

check("Exit D · mean_ answered", None if not exit_d_mean else len(exit_d_mean) > 25, True)
check("Exit D · strategy answered", None if not exit_d_strategy else len(exit_d_strategy) > 25, True)

print()
print("Mark yourself. A correct pair of answers contains both of these ideas:")
print("  - mean_ was created by fit, FROM THE DATA; the underscore is the convention that says so,")
print("    and it did not exist on the object before fit ran.")
print("  - strategy was CHOSEN BY YOU and stored by __init__; no data influenced it, so it needs")
print("    no marker -- it is a hyperparameter, readable before fit and unchanged by it.")

### What the exit ticket proved

- **A** - you can cross the boundary between a file and Python: open with `encoding`, read with
  `DictReader`, convert at the boundary, and raise a useful `FileNotFoundError` rather than a bare one.
- **B** - `["try", "finally"]`. The `return` inside `try` means `else` never runs, and `finally` runs
  anyway, past the `return`. If you predicted `["try", "else", "finally"]`, reread
  [Part 2](index.html#else-finally): `else` runs only when `try` completes **without returning**.
- **C** - the three defects were `entries = []` at class level (one list shared by every instance),
  `def add(minutes)` with no `self`, and `except Exception: pass` swallowing the evidence.
- **D** - the trailing underscore is the whole of scikit-learn's API documented in one character.

If all four passed, you are ready for Module 04.

## Summary and self-assessment

Check an item only when you can do it **without copying the lesson example**:

- [ ] I can read a traceback bottom-up and say what, where, and why.
- [ ] I can tell the crash site apart from the root cause, and explain the `average([])` example.
- [ ] I can read a chained traceback and say which block holds the origin.
- [ ] I can name all eleven common exceptions from their messages.
- [ ] I can explain the difference between `TypeError` and `ValueError` with an example of each.
- [ ] I can explain `UnboundLocalError` in terms of Module 02's scope rule.
- [ ] I can write `try`/`except` catching the narrowest exception I can name.
- [ ] I can explain why `except Exception: pass` is worse than no error handling at all.
- [ ] I can use `as error` to report an exception's type, message, and `args`.
- [ ] I can predict the printed order for `try`/`except`/`else`/`finally`, including past a `return`.
- [ ] I can raise a built-in exception to validate arguments at the top of a function.
- [ ] I can write a custom exception class and explain why two parents can be useful.
- [ ] I can use `raise ... from error` and explain what `__cause__` holds.
- [ ] I can say when `assert` is the wrong tool.
- [ ] I can explain why a relative path failed, using `Path.cwd()`.
- [ ] I can build paths with `pathlib` and create folders with `mkdir(parents=True, exist_ok=True)`.
- [ ] I can state what `"w"` does the instant a file is opened.
- [ ] I can explain why every `open()` needs `encoding="utf-8"`, and what mojibake looks like.
- [ ] I can read and write CSV with `DictReader`/`DictWriter`, and I know why `newline=""` is there.
- [ ] I can name what changes in a JSON round trip, and what raises instead.
- [ ] I can write a class with `__init__`, `self`, a method, and `__repr__`, and explain the
      shared-mutable-class-attribute bug.
- [ ] I can explain why `predict()` before `fit()` should raise rather than return `None`.

Twenty-two items. Anything unticked names its own next step, and every one of them maps to a numbered
section above.

## Where this goes next

Three things you built here are used directly, and soon:

- **`Vector2.dot()`** from Challenge A becomes geometry in Module 04, and a single NumPy operation in
  Module 07.
- **`MeanMinutesModel`** is a real baseline. Module 11 trains a regression and compares it against
  exactly this `score`, and a model that cannot beat a constant prediction is not a model.
- **Reading a traceback** is the skill you will use most often in every remaining module, because
  every library you are about to meet reports its problems this way.

Return to the [lesson self-check](index.html#self-check), then continue to
[Module 04 · Linear Algebra](../04-linear-algebra/index.html) - where your `Vector2.dot()` gets a
geometric meaning, and the underscore convention you just learned starts paying rent.